Stage 1: LoRA checkpoint-700 direct translator
        English/context → LoRA draft

Stage 2: LoRA router-corrector on top of the frozen LoRA-700 model
        English/context + LoRA draft → corrected Egyptian Arabic

This notebook no longer trains or evaluates an NTK-Mirror controller. The second stage is a PEFT/LoRA adapter trained as a router-corrector:
- useful but weak Arabic drafts are corrected/rewritten;
- garbage / wrong-language / empty / Chinese / mostly non-Arabic drafts are ignored and retranslated from source/context;
- already good drafts are kept close;
- output is only the final Egyptian Arabic translation.


### **Installations, Imports and Configurations**

In [1]:
# ============================================================
# Cell 1 — Installations for Qwen3-4B LoRA router-corrector
# ============================================================
#
# Goal:
#   Use an already fine-tuned Qwen3-4B LoRA adapter as the frozen
#   first-stage translator, then train a second PEFT/LoRA adapter as
#   a router-corrector on top of it.
#
# This cell installs only the packages needed for:
#   - loading Qwen3-4B in 4-bit
#   - loading the frozen first-stage PEFT/LoRA adapter
#   - training the second-stage PEFT/LoRA router-corrector
#   - computing BLEU, spBLEU, chrF, chrF++
#   - later computing E5 semantic similarity
#
# NTK-Mirror is intentionally not installed or used in this notebook.
# ============================================================

import importlib.util
import subprocess
import sys
from importlib.metadata import version as pkg_version, PackageNotFoundError


def is_installed(import_name: str) -> bool:
    return importlib.util.find_spec(import_name) is not None


def installed_version(package_name: str):
    try:
        return pkg_version(package_name)
    except PackageNotFoundError:
        return None


# package_name_on_pip : import_name_in_python
required = {
    "transformers": "transformers",
    "peft": "peft",
    "accelerate": "accelerate",
    "bitsandbytes": "bitsandbytes",
    "datasets": "datasets",
    "sacrebleu": "sacrebleu",
    "sentencepiece": "sentencepiece",
    "packaging": "packaging",
    "tqdm": "tqdm",
    "pandas": "pandas",
    "numpy": "numpy",
    "openpyxl": "openpyxl",
    "sentence-transformers": "sentence_transformers",
}

missing = [
    pip_name
    for pip_name, import_name in required.items()
    if not is_installed(import_name)
]

print("Missing packages:", missing)

if missing:
    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        *missing,
    ]
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)
else:
    print("All required regular packages are already installed.")


# ------------------------------------------------------------
# Qwen3 + adapter stacking require recent Transformers/PEFT
# ------------------------------------------------------------

from packaging.version import parse as parse_version

transformers_version = installed_version("transformers")
peft_version = installed_version("peft")

print("Current transformers:", transformers_version)
print("Current peft:", peft_version)

upgrade_cmd = None

if transformers_version is None or parse_version(transformers_version) < parse_version("4.51.0"):
    upgrade_cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "transformers>=4.51.0",
        "peft>=0.11.0",
        "accelerate",
    ]
elif peft_version is None or parse_version(peft_version) < parse_version("0.11.0"):
    upgrade_cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        "peft>=0.11.0",
        "accelerate",
    ]

if upgrade_cmd is not None:
    print("Upgrading packages for Qwen3 + multi-adapter PEFT support:")
    print("Running:", " ".join(upgrade_cmd))
    subprocess.check_call(upgrade_cmd)
else:
    print("transformers / peft versions look OK.")


print("\nCell 1 finished.")
print("If Colab asks for a runtime restart after installation, restart and rerun Cell 1.")


Missing packages: ['bitsandbytes', 'sacrebleu']
Running: /usr/bin/python3 -m pip install -q --no-cache-dir bitsandbytes sacrebleu
Current transformers: 5.12.0
Current peft: 0.19.1
transformers / peft versions look OK.

Cell 1 finished.
If Colab asks for a runtime restart after installation, restart and rerun Cell 1.


In [2]:
# ============================================================
# Cell 1B — Environment check
# ============================================================
#
# Checks:
#   - Python / PyTorch / CUDA
#   - GPU name and VRAM
#   - important package versions
#   - PEFT multi-adapter support needed by the LoRA router-corrector
#
# This notebook is expected to run on Colab T4-like hardware.
# ============================================================

import os
import sys
import random
import platform
import subprocess
from importlib.metadata import version as pkg_version, PackageNotFoundError

import numpy as np
import torch


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seed:", SEED)


# ------------------------------------------------------------
# Basic environment
# ------------------------------------------------------------

print("\n================ Python / System ================")
print("Python:", sys.version)
print("Platform:", platform.platform())
print("Executable:", sys.executable)


# ------------------------------------------------------------
# PyTorch / CUDA
# ------------------------------------------------------------

print("\n================ PyTorch / CUDA ================")
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    reserved_gb = torch.cuda.memory_reserved(0) / 1024**3
    allocated_gb = torch.cuda.memory_allocated(0) / 1024**3

    print("GPU:", gpu_name)
    print(f"Total VRAM: {total_vram_gb:.2f} GB")
    print(f"Reserved VRAM: {reserved_gb:.2f} GB")
    print(f"Allocated VRAM: {allocated_gb:.2f} GB")
else:
    print("WARNING: No GPU detected. This notebook will be too slow without CUDA.")


# ------------------------------------------------------------
# nvidia-smi
# ------------------------------------------------------------

print("\n================ nvidia-smi ================")
try:
    result = subprocess.run(
        ["nvidia-smi"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
except Exception as e:
    print("Could not run nvidia-smi:", repr(e))


# ------------------------------------------------------------
# Package versions
# ------------------------------------------------------------

def safe_version(package_name):
    try:
        return pkg_version(package_name)
    except PackageNotFoundError:
        return "NOT INSTALLED"


packages_to_check = [
    "transformers",
    "peft",
    "accelerate",
    "bitsandbytes",
    "datasets",
    "sacrebleu",
    "sentence-transformers",
    "sentencepiece",
]

print("\n================ Package versions ================")
for pkg in packages_to_check:
    print(f"{pkg:24s}: {safe_version(pkg)}")


# ------------------------------------------------------------
# Import checks
# ------------------------------------------------------------

print("\n================ Import checks ================")

try:
    import transformers
    print("transformers import: OK")
except Exception as e:
    print("transformers import failed:", repr(e))

try:
    import peft
    print("peft import: OK")
except Exception as e:
    print("peft import failed:", repr(e))

try:
    import bitsandbytes as bnb
    print("bitsandbytes import: OK")
except Exception as e:
    print("bitsandbytes import failed:", repr(e))


# ------------------------------------------------------------
# TF32 settings
# ------------------------------------------------------------

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print("\nTF32 enabled for CUDA matmul/cudnn where supported.")


print("\nCell 1B finished.")


Seed: 42

================ Python / System ================
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35
Executable: /usr/bin/python3

================ PyTorch / CUDA ================
Torch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
Total VRAM: 14.56 GB
Reserved VRAM: 0.00 GB
Allocated VRAM: 0.00 GB

================ nvidia-smi ================
Tue Jun 23 15:41:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=

In [3]:
# ============================================================
# Cell 2 — Mount Google Drive and define project paths
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/alexandria_qwen35_sft")

DATA_DIR = PROJECT_DIR / "prepared_data"
RUNS_DIR = PROJECT_DIR / "runs"
ADAPTER_DIR = PROJECT_DIR / "final_adapters"
PRED_DIR = PROJECT_DIR / "predictions"
SEMANTIC_DIR = PROJECT_DIR / "semantic_similarity_e5_large"

for p in [PROJECT_DIR, DATA_DIR, RUNS_DIR, ADAPTER_DIR, PRED_DIR, SEMANTIC_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR:", DATA_DIR)
print("RUNS_DIR:", RUNS_DIR)
print("ADAPTER_DIR:", ADAPTER_DIR)
print("PRED_DIR:", PRED_DIR)
print("SEMANTIC_DIR:", SEMANTIC_DIR)

Mounted at /content/drive
PROJECT_DIR: /content/drive/MyDrive/alexandria_qwen35_sft
DATA_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/prepared_data
RUNS_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/runs
ADAPTER_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/final_adapters
PRED_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/predictions
SEMANTIC_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/semantic_similarity_e5_large


In [4]:
# ============================================================
# Cell 3 — Main experiment configuration
# Qwen3-4B LoRA checkpoint-700 + second-stage LoRA router-corrector
# Full-train draft cache + balanced bad_arabic / garbage / retain mixture
# Full eval version
# ============================================================

from pathlib import Path
import json
import random
import numpy as np
import torch

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Base model + first-stage LoRA adapter
# ------------------------------------------------------------

MODEL_NAME = "Qwen/Qwen3-4B-Base"
LOAD_IN_4BIT = True

LORA_EXPERIMENT_NAME = (
    "qwen3_4b_alexandria_eg_only_context3_"
    "complete2shot_all_group_r16_10epochs"
)

LORA_RUN_DIR = RUNS_DIR / LORA_EXPERIMENT_NAME

# Best first-stage LoRA checkpoint.
MANUAL_LORA_BEST_STEP = 700

if MANUAL_LORA_BEST_STEP is not None:
    LORA_ADAPTER_PATH = LORA_RUN_DIR / f"checkpoint-{MANUAL_LORA_BEST_STEP}"
else:
    LORA_ADAPTER_PATH = None

# Existing Cell 8 loads the first-stage adapter without an explicit adapter name,
# so PEFT names it "default". Keep this unless you also rewrite Cell 8.
FIRST_STAGE_ADAPTER_NAME = "default"
CORRECTOR_ADAPTER_NAME = "router_corrector"
CORRECTOR_ADAPTER_SAVE_SUBDIR = "router_corrector_adapter"

# ------------------------------------------------------------
# Dataset setup
# ------------------------------------------------------------

DATASET_NAME = "UBC-NLP/alexandria"

SELECTED_CONFIGS_MODE = "EG_ONLY"  # "EG_ONLY", "ALL", or "MANUAL"
MANUAL_CONFIGS = ["EG"]

MAX_CONTEXT_TURNS = 3
USE_PREVIOUS_ENGLISH_CONTEXT = True
USE_METADATA = True

# ------------------------------------------------------------
# Prompting setup
# ------------------------------------------------------------

USE_FEW_SHOTS = True
N_FEW_SHOTS = 2
MAX_FEW_SHOT_EXAMPLE_CHARS = 450

# Corrector prompt is longer because it includes source/context + LoRA draft.
MAX_SEQ_LENGTH = 2048
MAX_NEW_TOKENS = 120

# ------------------------------------------------------------
# Second-stage LoRA router-corrector setup
# ------------------------------------------------------------

STAGE2_MODE = "lora_router_corrector"

# Full train draft generation. No arbitrary cap.
CORRECTOR_TRAIN_CANDIDATE_LIMIT = None

# Full eval draft generation. No arbitrary cap.
CORRECTOR_EVAL_LIMIT = None

# Used to define low-quality Arabic drafts from the full-train draft cache.
CORRECTOR_BAD_FRACTION = 0.30
CORRECTOR_SELECTION_METRIC = "chrF++"  # "chrF++" or "spBLEU"

# Balanced router-corrector training mixture.
# The total requested training rows defaults to full-train size.
CORRECTOR_MIX_BAD_ARABIC = 0.50
CORRECTOR_MIX_GARBAGE = 0.25
CORRECTOR_MIX_RETAIN = 0.25
CORRECTOR_TRAIN_TARGET_TOTAL = None  # None means len(full train draft cache)

# Script / garbage diagnostics.
ARABIC_RATIO_MIN_FOR_ARABIC = 0.30
NON_ARABIC_RATIO_FOR_GARBAGE = 0.50

# Second-stage PEFT/LoRA hyperparameters.
CORRECTOR_LORA_R = 8       # Change to 4 if Colab T4 gets OOM
CORRECTOR_LORA_ALPHA = 16
CORRECTOR_LORA_DROPOUT = 0.05

CORRECTOR_LORA_TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

# ------------------------------------------------------------
# Training hyperparameters
# ------------------------------------------------------------

CORRECTOR_NUM_EPOCHS = 3
NUM_EPOCHS = CORRECTOR_NUM_EPOCHS  # kept for older print/helpers

PER_DEVICE_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8

CORRECTOR_LR = 5e-5
LEARNING_RATE = CORRECTOR_LR  # kept for older print/helpers

WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.0

LOGGING_STEPS = 10
EVAL_STEPS = 50
SAVE_STEPS = 50
SAVE_TOTAL_LIMIT = 20

# Optional small NLL sanity eval only.
# Generation metrics are still Cell 16/18/19.
EVAL_NLL_LIMIT = 64

DRAFT_SAVE_EVERY = 25
PRED_SAVE_EVERY = 25

# ------------------------------------------------------------
# Metric checkpoint sweep
# ------------------------------------------------------------

SWEEP_EVAL_LIMIT = None

# Cell 16 will ignore steps that do not exist.
# If empty after training, it automatically sweeps all found LoRA-corrector checkpoints.
SWEEP_STEPS = [50, 100, 150, 200, 300, 400]

PRIMARY_SELECTION_METRIC = "chrF++"
SECONDARY_SELECTION_METRIC = "spBLEU"

# ------------------------------------------------------------
# Experiment name
# ------------------------------------------------------------

LORA_STEP_FOR_NAME = (
    MANUAL_LORA_BEST_STEP
    if MANUAL_LORA_BEST_STEP is not None
    else "auto"
)

EVAL_TAG_FOR_NAME = (
    "fulleval"
    if CORRECTOR_EVAL_LIMIT is None
    else f"eval{CORRECTOR_EVAL_LIMIT}"
)

TRAIN_TAG_FOR_NAME = "fulltrain"

metric_tag = CORRECTOR_SELECTION_METRIC.lower().replace("+", "p")
bad_frac_tag = str(CORRECTOR_BAD_FRACTION).replace(".", "p")

EXPERIMENT_NAME = (
    f"qwen3_4b_lora_step{LORA_STEP_FOR_NAME}_"
    f"alexandria_eg_only_context3_complete2shot_"
    f"LORA_ROUTER_CORRECTOR_draft2gold_"
    f"{TRAIN_TAG_FOR_NAME}_mix_badarabic_garbage_retain_"
    f"worst{bad_frac_tag}_{metric_tag}_"
    f"{EVAL_TAG_FOR_NAME}_"
    f"r{CORRECTOR_LORA_R}_alpha{CORRECTOR_LORA_ALPHA}_"
    f"drop{str(CORRECTOR_LORA_DROPOUT).replace('.', 'p')}_"
    f"lr{str(CORRECTOR_LR).replace('.', 'p')}_"
    f"{CORRECTOR_NUM_EPOCHS}epochs"
)

OUTPUT_DIR = RUNS_DIR / EXPERIMENT_NAME
CORRECTOR_DIR = ADAPTER_DIR / EXPERIMENT_NAME

# Backward-compatible name for old cells that wrote final artifacts there.
CONTROLLER_DIR = CORRECTOR_DIR

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CORRECTOR_DIR.mkdir(parents=True, exist_ok=True)

print("MODEL_NAME:", MODEL_NAME)
print("LORA_EXPERIMENT_NAME:", LORA_EXPERIMENT_NAME)
print("LORA_RUN_DIR:", LORA_RUN_DIR)
print("MANUAL_LORA_BEST_STEP:", MANUAL_LORA_BEST_STEP)
print("LORA_ADAPTER_PATH:", LORA_ADAPTER_PATH)

print("\nEXPERIMENT_NAME:", EXPERIMENT_NAME)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("CORRECTOR_DIR:", CORRECTOR_DIR)

print("\nSecond-stage config:")
print("  STAGE2_MODE:", STAGE2_MODE)
print("  FIRST_STAGE_ADAPTER_NAME:", FIRST_STAGE_ADAPTER_NAME)
print("  CORRECTOR_ADAPTER_NAME:", CORRECTOR_ADAPTER_NAME)
print("  CORRECTOR_LORA_R:", CORRECTOR_LORA_R)
print("  CORRECTOR_LORA_ALPHA:", CORRECTOR_LORA_ALPHA)
print("  CORRECTOR_LORA_DROPOUT:", CORRECTOR_LORA_DROPOUT)
print("  CORRECTOR_LR:", CORRECTOR_LR)
print("  CORRECTOR_NUM_EPOCHS:", CORRECTOR_NUM_EPOCHS)
print("  MAX_SEQ_LENGTH:", MAX_SEQ_LENGTH)
print("  MAX_NEW_TOKENS:", MAX_NEW_TOKENS)

print("\nCorrector data config:")
print("  CORRECTOR_TRAIN_CANDIDATE_LIMIT:", CORRECTOR_TRAIN_CANDIDATE_LIMIT)
print("  CORRECTOR_EVAL_LIMIT:", CORRECTOR_EVAL_LIMIT)
print("  CORRECTOR_BAD_FRACTION:", CORRECTOR_BAD_FRACTION)
print("  CORRECTOR_SELECTION_METRIC:", CORRECTOR_SELECTION_METRIC)
print("  CORRECTOR_MIX_BAD_ARABIC:", CORRECTOR_MIX_BAD_ARABIC)
print("  CORRECTOR_MIX_GARBAGE:", CORRECTOR_MIX_GARBAGE)
print("  CORRECTOR_MIX_RETAIN:", CORRECTOR_MIX_RETAIN)
print("  CORRECTOR_TRAIN_TARGET_TOTAL:", CORRECTOR_TRAIN_TARGET_TOTAL)

print("\nStage-1 draft cache reuse:")
print("  Cell 9B will first copy existing LoRA-700 train/eval draft caches from the previous NTK experiment, if found.")
print("  New router-corrector outputs still stay under OUTPUT_DIR:", OUTPUT_DIR)

MODEL_NAME: Qwen/Qwen3-4B-Base
LORA_EXPERIMENT_NAME: qwen3_4b_alexandria_eg_only_context3_complete2shot_all_group_r16_10epochs
LORA_RUN_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_alexandria_eg_only_context3_complete2shot_all_group_r16_10epochs
MANUAL_LORA_BEST_STEP: 700
LORA_ADAPTER_PATH: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_alexandria_eg_only_context3_complete2shot_all_group_r16_10epochs/checkpoint-700

EXPERIMENT_NAME: qwen3_4b_lora_step700_alexandria_eg_only_context3_complete2shot_LORA_ROUTER_CORRECTOR_draft2gold_fulltrain_mix_badarabic_garbage_retain_worst0p3_chrfpp_fulleval_r8_alpha16_drop0p05_lr5e-05_3epochs
OUTPUT_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_lora_step700_alexandria_eg_only_context3_complete2shot_LORA_ROUTER_CORRECTOR_draft2gold_fulltrain_mix_badarabic_garbage_retain_worst0p3_chrfpp_fulleval_r8_alpha16_drop0p05_lr5e-05_3epochs
CORRECTOR_DIR: /content/drive/MyDrive/alexandria_qwen35_sft/final_adapters/q

In [5]:
# ============================================================
# Cell 3B — Resolve best LoRA adapter checkpoint
# ============================================================

import json
from pathlib import Path

def checkpoint_step(path):
    path = Path(path)
    name = path.name

    if not name.startswith("checkpoint-"):
        return -1

    try:
        return int(name.replace("checkpoint-", ""))
    except Exception:
        return -1


def read_json_safe(path):
    path = Path(path)

    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception as e:
        print(f"WARNING: could not read JSON file: {path}")
        print("Reason:", repr(e))
        return None


def find_trainer_state_files(run_dir, checkpoint_dir=None):
    run_dir = Path(run_dir)
    candidates = []

    if checkpoint_dir is not None:
        checkpoint_dir = Path(checkpoint_dir)
        candidates.extend([
            checkpoint_dir / "trainer_state.json",
            checkpoint_dir.parent / "trainer_state.json",
        ])

    candidates.extend([
        run_dir / "trainer_state.json",
        run_dir / "trainer_state_best.json",
    ])

    # Also search recursively as fallback.
    candidates.extend(list(run_dir.rglob("trainer_state.json")))

    # Deduplicate while preserving order.
    seen = set()
    unique = []

    for p in candidates:
        p = Path(p)
        key = str(p)

        if key in seen:
            continue

        seen.add(key)

        if p.exists():
            unique.append(p)

    return unique


def extract_eval_loss_for_step(state, step):
    if state is None:
        return None

    step = int(step)

    matched_losses = []

    for item in state.get("log_history", []):
        try:
            item_step = int(item.get("step", -1))
        except Exception:
            continue

        if item_step != step:
            continue

        if "eval_loss" in item:
            matched_losses.append(float(item["eval_loss"]))
        elif "eval_nll" in item:
            matched_losses.append(float(item["eval_nll"]))

    if matched_losses:
        return matched_losses[-1]

    return None


def find_best_lora_checkpoint(output_dir):
    output_dir = Path(output_dir)

    state_files = find_trainer_state_files(output_dir)

    if not state_files:
        raise FileNotFoundError(f"No trainer_state.json found under: {output_dir}")

    readable_states = []

    for sf in state_files:
        state = read_json_safe(sf)

        if state is None:
            continue

        global_step = int(state.get("global_step", -1))
        readable_states.append((global_step, sf, state))

    if not readable_states:
        raise RuntimeError("Could not read a valid trainer_state.json")

    # Prefer the trainer_state with the largest global_step.
    readable_states = sorted(readable_states, key=lambda x: x[0], reverse=True)

    _, best_state_file, best_state = readable_states[0]

    official_best = best_state.get("best_model_checkpoint", None)
    official_metric = best_state.get("best_metric", None)

    if official_best is not None:
        official_best_path = Path(official_best)

        if not official_best_path.exists():
            official_best_path = output_dir / official_best_path.name

        if official_best_path.exists():
            return (
                official_best_path,
                checkpoint_step(official_best_path),
                official_metric,
                best_state_file,
            )

    eval_rows = []

    for item in best_state.get("log_history", []):
        if "step" not in item:
            continue

        if "eval_loss" in item:
            eval_rows.append({
                "step": int(item["step"]),
                "eval_loss": float(item["eval_loss"]),
            })
        elif "eval_nll" in item:
            eval_rows.append({
                "step": int(item["step"]),
                "eval_loss": float(item["eval_nll"]),
            })

    if not eval_rows:
        raise RuntimeError("No eval_loss/eval_nll records found in LoRA trainer_state.json")

    best_row = min(eval_rows, key=lambda x: x["eval_loss"])
    best_step = int(best_row["step"])
    best_path = output_dir / f"checkpoint-{best_step}"

    if not best_path.exists():
        raise FileNotFoundError(f"Inferred best LoRA checkpoint missing: {best_path}")

    return best_path, best_step, best_row["eval_loss"], best_state_file


def find_lora_metadata_for_manual_checkpoint(run_dir, adapter_path, step):
    run_dir = Path(run_dir)
    adapter_path = Path(adapter_path)
    step = int(step)

    state_files = find_trainer_state_files(run_dir, checkpoint_dir=adapter_path)

    best_state_file = None
    best_eval_loss = None

    for sf in state_files:
        state = read_json_safe(sf)

        if state is None:
            continue

        # First: try exact eval loss for this step.
        eval_loss = extract_eval_loss_for_step(state, step)

        if eval_loss is not None:
            return eval_loss, sf

        # Second: if trainer_state says this is best_model_checkpoint, use best_metric.
        official_best = state.get("best_model_checkpoint", None)

        if official_best is not None:
            official_best_path = Path(official_best)

            official_step = checkpoint_step(official_best_path)

            if official_step == step and state.get("best_metric") is not None:
                return float(state["best_metric"]), sf

        # Keep first readable state file as metadata source even if eval not found.
        if best_state_file is None:
            best_state_file = sf

    return best_eval_loss, best_state_file


# ------------------------------------------------------------
# Resolve adapter path
# ------------------------------------------------------------

if LORA_ADAPTER_PATH is None:
    (
        LORA_ADAPTER_PATH,
        LORA_BEST_STEP,
        LORA_BEST_EVAL_LOSS,
        LORA_STATE_FILE,
    ) = find_best_lora_checkpoint(LORA_RUN_DIR)

else:
    LORA_ADAPTER_PATH = Path(LORA_ADAPTER_PATH)
    LORA_BEST_STEP = checkpoint_step(LORA_ADAPTER_PATH)

    if LORA_BEST_STEP < 0:
        if MANUAL_LORA_BEST_STEP is None:
            raise ValueError(
                f"Could not infer checkpoint step from LORA_ADAPTER_PATH={LORA_ADAPTER_PATH}. "
                "Set MANUAL_LORA_BEST_STEP explicitly."
            )

        LORA_BEST_STEP = int(MANUAL_LORA_BEST_STEP)

    (
        LORA_BEST_EVAL_LOSS,
        LORA_STATE_FILE,
    ) = find_lora_metadata_for_manual_checkpoint(
        run_dir=LORA_RUN_DIR,
        adapter_path=LORA_ADAPTER_PATH,
        step=LORA_BEST_STEP,
    )


# ------------------------------------------------------------
# Validate adapter path/files
# ------------------------------------------------------------

LORA_ADAPTER_PATH = Path(LORA_ADAPTER_PATH)

if not LORA_ADAPTER_PATH.exists():
    raise FileNotFoundError(
        f"LoRA adapter path does not exist:\n{LORA_ADAPTER_PATH}\n\n"
        f"Check LORA_EXPERIMENT_NAME and MANUAL_LORA_BEST_STEP in Cell 3."
    )

adapter_config = LORA_ADAPTER_PATH / "adapter_config.json"
adapter_safetensors = LORA_ADAPTER_PATH / "adapter_model.safetensors"
adapter_bin = LORA_ADAPTER_PATH / "adapter_model.bin"

if not adapter_config.exists():
    raise FileNotFoundError(f"Missing adapter_config.json:\n{adapter_config}")

if not adapter_safetensors.exists() and not adapter_bin.exists():
    raise FileNotFoundError(
        "Missing LoRA adapter weights. Expected one of:\n"
        f"  {adapter_safetensors}\n"
        f"  {adapter_bin}"
    )

print("LoRA adapter files look OK.")

print("Resolved LoRA adapter:")
print("  path:", LORA_ADAPTER_PATH)
print("  step:", LORA_BEST_STEP)

if LORA_BEST_EVAL_LOSS is None:
    print("  eval_loss: not found / not required")
else:
    print("  eval_loss:", LORA_BEST_EVAL_LOSS)

if LORA_STATE_FILE is None:
    print("  state_file: not found / adapter-only checkpoint is OK")
else:
    print("  state_file:", LORA_STATE_FILE)

LoRA adapter files look OK.
Resolved LoRA adapter:
  path: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_alexandria_eg_only_context3_complete2shot_all_group_r16_10epochs/checkpoint-700
  step: 700
  eval_loss: 1.6058480739593506
  state_file: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_alexandria_eg_only_context3_complete2shot_all_group_r16_10epochs/checkpoint-700/trainer_state.json


### **Dataset Preparations**

In [6]:
# ============================================================
# Cell 4 — Load Alexandria configs
# ============================================================

from datasets import load_dataset, get_dataset_config_names
import pandas as pd

DATASET_NAME = "UBC-NLP/alexandria"

available_configs = get_dataset_config_names(DATASET_NAME)
print("Available configs:")
print(available_configs)

if SELECTED_CONFIGS_MODE == "EG_ONLY":
    selected_configs = ["EG"] if "EG" in available_configs else [available_configs[0]]

elif SELECTED_CONFIGS_MODE == "ALL":
    selected_configs = available_configs

elif SELECTED_CONFIGS_MODE == "MANUAL":
    selected_configs = MANUAL_CONFIGS
    missing = [c for c in selected_configs if c not in available_configs]
    if missing:
        raise ValueError(f"Unavailable configs: {missing}")

else:
    raise ValueError("SELECTED_CONFIGS_MODE must be EG_ONLY, ALL, or MANUAL.")

print("\nSelected configs:")
print(selected_configs)

loaded = {}

for cfg in selected_configs:
    print(f"\nLoading config: {cfg}")

    ds_train = load_dataset(DATASET_NAME, name=cfg, split="train")
    ds_test = load_dataset(DATASET_NAME, name=cfg, split="test")

    loaded[cfg] = {
        "train": ds_train,
        "test": ds_test,
    }

    print("Train:", ds_train)
    print("Test:", ds_test)
    print("Keys:", ds_train[0].keys())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/24.5k [00:00<?, ?B/s]

Available configs:
['EG', 'JO', 'LB', 'LY', 'MA', 'MR', 'OM', 'PS', 'SA', 'SD', 'SY', 'TN', 'YE']

Selected configs:
['EG']

Loading config: EG


EG/train-00000-of-00001.parquet:   0%|          | 0.00/496k [00:00<?, ?B/s]

EG/test-00000-of-00001.parquet:   0%|          | 0.00/196k [00:00<?, ?B/s]

EG/dev-00000-of-00001.parquet:   0%|          | 0.00/183k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/982 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/366 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/352 [00:00<?, ? examples/s]

Train: Dataset({
    features: ['conv_id', 'country', 'domain', 'dialect', 'participants', 'english_conversation', 'dialectal_conversation', 'translator_id', 'reviewer_id'],
    num_rows: 982
})
Test: Dataset({
    features: ['conv_id', 'country', 'domain', 'dialect', 'participants', 'english_conversation', 'dialectal_conversation', 'translator_id', 'reviewer_id'],
    num_rows: 366
})
Keys: dict_keys(['conv_id', 'country', 'domain', 'dialect', 'participants', 'english_conversation', 'dialectal_conversation', 'translator_id', 'reviewer_id'])


In [7]:
# ============================================================
# Cell 5 — Flatten Alexandria conversations
# ============================================================

def safe_get(row, keys, default=""):
    for k in keys:
        if isinstance(row, dict) and k in row and row[k] is not None:
            return row[k]
    return default

def turn_text(turn):
    if isinstance(turn, dict):
        for k in ["text", "sentence", "utterance", "content", "value"]:
            if k in turn and turn[k] is not None:
                return str(turn[k]).strip()
        return str(turn).strip()
    return str(turn).strip()

def turn_field(turn, keys, default=""):
    if isinstance(turn, dict):
        for k in keys:
            if k in turn and turn[k] is not None:
                return str(turn[k]).strip()
    return default

def normalize_list(x):
    if x is None:
        return []
    if isinstance(x, list):
        return x
    return list(x)

def flatten_alexandria_split(ds, cfg_name, split_name, max_context_turns=3):
    records = []

    for conv_idx, row in enumerate(ds):
        english_conv = normalize_list(row["english_conversation"])
        dialect_conv = normalize_list(row["dialectal_conversation"])

        n = min(len(english_conv), len(dialect_conv))

        country = safe_get(row, ["country", "country_code"], cfg_name)
        dialect = safe_get(row, ["dialect", "dialect_label", "subdialect", "city", "variety"], "")
        domain = safe_get(row, ["domain", "topic"], "")
        persona = safe_get(row, ["persona", "roles", "speaker_roles"], "")
        conv_id = safe_get(row, ["conversation_id", "id", "dialogue_id"], f"{cfg_name}_{split_name}_{conv_idx}")

        for i in range(n):
            en_turn = english_conv[i]
            ar_turn = dialect_conv[i]

            source_text = turn_text(en_turn)
            target_text = turn_text(ar_turn)

            if not source_text or not target_text:
                continue

            prev_start = max(0, i - max_context_turns)
            prev_en_turns = english_conv[prev_start:i]

            previous_context = []
            for j, t in enumerate(prev_en_turns, start=prev_start):
                previous_context.append({
                    "turn_id": j,
                    "speaker": turn_field(t, ["speaker", "role", "speaker_role"], ""),
                    "direction": turn_field(t, ["direction", "gender_direction", "speaker_addressee_gender"], ""),
                    "text": turn_text(t),
                })

            records.append({
                "source_id": f"{cfg_name}_{split_name}_{conv_id}_{i}",
                "config": cfg_name,
                "split": split_name,
                "conversation_id": conv_id,
                "turn_id": i,

                "country": country,
                "dialect": dialect,
                "domain": domain,
                "persona": persona,

                "speaker": turn_field(en_turn, ["speaker", "role", "speaker_role"], ""),
                "gender_direction": turn_field(en_turn, ["direction", "gender_direction", "speaker_addressee_gender"], ""),

                "previous_english_turns": previous_context,
                "source_text": source_text,
                "target_arabic": target_text,
            })

    return records

train_records = []
eval_records = []

for cfg in selected_configs:
    train_records.extend(
        flatten_alexandria_split(
            loaded[cfg]["train"],
            cfg_name=cfg,
            split_name="train",
            max_context_turns=MAX_CONTEXT_TURNS,
        )
    )

    eval_records.extend(
        flatten_alexandria_split(
            loaded[cfg]["test"],
            cfg_name=cfg,
            split_name="test",
            max_context_turns=MAX_CONTEXT_TURNS,
        )
    )

train_df = pd.DataFrame(train_records)
eval_df = pd.DataFrame(eval_records)

print("Train shape:", train_df.shape)
print("Eval shape:", eval_df.shape)

print("\nTrain configs:")
print(train_df["config"].value_counts())

print("\nEval configs:")
print(eval_df["config"].value_counts())

train_jsonl = DATA_DIR / f"alexandria_train_{SELECTED_CONFIGS_MODE.lower()}_context{MAX_CONTEXT_TURNS}.jsonl"
eval_jsonl = DATA_DIR / f"alexandria_eval_{SELECTED_CONFIGS_MODE.lower()}_context{MAX_CONTEXT_TURNS}.jsonl"

train_df.to_json(train_jsonl, orient="records", lines=True, force_ascii=False)
eval_df.to_json(eval_jsonl, orient="records", lines=True, force_ascii=False)

print("\nSaved:")
print(train_jsonl)
print(eval_jsonl)

display(train_df.head())

Train shape: (3108, 14)
Eval shape: (1118, 14)

Train configs:
config
EG    3108
Name: count, dtype: int64

Eval configs:
config
EG    1118
Name: count, dtype: int64

Saved:
/content/drive/MyDrive/alexandria_qwen35_sft/prepared_data/alexandria_train_eg_only_context3.jsonl
/content/drive/MyDrive/alexandria_qwen35_sft/prepared_data/alexandria_eval_eg_only_context3.jsonl


,source_id,config,split,conversation_id,turn_id,country,dialect,domain,persona,speaker,gender_direction,previous_english_turns,source_text,target_arabic
0,EG_train_EG_train_0_0,EG,train,EG_train_0,0,EG,Egyptian Arabic (Cairene) Dialect,Agriculture and farming,,Wholesale Buyer,male -> female,[],Good morning. I'm looking to source 10 tons of...,صباح الخير، عايز عشرة طن من الخرشوف الكويس للت...
1,EG_train_EG_train_0_1,EG,train,EG_train_0,1,EG,Egyptian Arabic (Cairene) Dialect,Agriculture and farming,,Wholesale Seller,female -> male,"[{'turn_id': 0, 'speaker': 'Wholesale Buyer', ...",Good morning to you. You heard correctly. My a...,صباح النور،سمعك مظبوط،الخرشوف بتاعي من أحسن ال...
2,EG_train_EG_train_0_2,EG,train,EG_train_0,2,EG,Egyptian Arabic (Cairene) Dialect,Agriculture and farming,,Wholesale Buyer,male -> female,"[{'turn_id': 0, 'speaker': 'Wholesale Buyer', ...","Excellent. Yes, please show me. I need them to...",ممتاز، لو سمحتي وريني، عايزه بمقاس واحد ومافيه...
3,EG_train_EG_train_0_3,EG,train,EG_train_0,3,EG,Egyptian Arabic (Cairene) Dialect,Agriculture and farming,,Wholesale Seller,female -> male,"[{'turn_id': 0, 'speaker': 'Wholesale Buyer', ...",Don't you worry. You will be very satisfied. M...,متخافش، هتنبسط جدا، سمعتي جاية من الحاجة الكويسة.
4,EG_train_EG_train_1_0,EG,train,EG_train_1,0,EG,Egyptian Arabic (Cairene) Dialect,Agriculture and farming,,Farmer,female -> female,[],I usually use the regular granular fertilizer....,أنا عادة بستخدم السماد العادي الحبيبات. ايه فا...


In [8]:
# ============================================================
# Cell 6 — Build prompts/messages with complete 2-shot examples
# Must match the Qwen3-4B LoRA run style
# ============================================================

from datasets import Dataset
import hashlib
import pandas as pd

SYSTEM_PROMPT = (
    "You are a professional machine translation system. "
    "Translate the current English dialogue turn into natural dialectal Arabic. "
    "Use the provided training examples only as style and dialect guidance. "
    "Return only the translation, without explanation."
)

def build_context(previous_turns):
    if not USE_PREVIOUS_ENGLISH_CONTEXT or not previous_turns:
        return "No previous context."

    lines = []
    for i, t in enumerate(previous_turns, start=1):
        speaker = t.get("speaker", "")
        text = t.get("text", "")

        if speaker:
            lines.append(f"{i}. {speaker}: {text}")
        else:
            lines.append(f"{i}. {text}")

    return "\n".join(lines)

def build_metadata_block(row):
    if not USE_METADATA:
        return "No metadata."

    fields = [
        ("Country/config", row.get("config", "")),
        ("Target dialect", row.get("dialect", "")),
        ("Domain", row.get("domain", "")),
        ("Persona/Roles", row.get("persona", "")),
        ("Current speaker", row.get("speaker", "")),
        ("Speaker-to-addressee gender direction", row.get("gender_direction", "")),
    ]

    lines = []
    for k, v in fields:
        v = str(v).strip()
        if v:
            lines.append(f"{k}: {v}")

    return "\n".join(lines) if lines else "No metadata."

def deterministic_seed_from_id(source_id, base_seed=SEED):
    raw = f"{source_id}_{base_seed}".encode("utf-8")
    return int(hashlib.md5(raw).hexdigest()[:8], 16)

def select_two_shots_from_train(row, train_pool, n=N_FEW_SHOTS):
    if not USE_FEW_SHOTS or n <= 0:
        return []

    row_source_id = str(row.get("source_id", ""))
    row_config = str(row.get("config", ""))
    row_domain = str(row.get("domain", ""))

    pool = train_pool.copy()
    pool["source_id"] = pool["source_id"].astype(str)
    pool = pool[pool["source_id"] != row_source_id].copy()

    if len(pool) == 0:
        return []

    pool["fewshot_total_chars"] = (
        pool["source_text"].astype(str).str.len()
        + pool["target_arabic"].astype(str).str.len()
    )

    short_pool = pool[pool["fewshot_total_chars"] <= MAX_FEW_SHOT_EXAMPLE_CHARS].copy()

    same_config_domain_short = short_pool[
        (short_pool["config"].astype(str) == row_config)
        & (short_pool["domain"].astype(str) == row_domain)
    ]

    same_config_short = short_pool[
        short_pool["config"].astype(str) == row_config
    ]

    same_config_domain = pool[
        (pool["config"].astype(str) == row_config)
        & (pool["domain"].astype(str) == row_domain)
    ]

    same_config = pool[
        pool["config"].astype(str) == row_config
    ]

    candidate_pools = [
        same_config_domain_short,
        same_config_short,
        short_pool,
        same_config_domain,
        same_config,
        pool,
    ]

    candidates = None
    for candidate_pool in candidate_pools:
        if len(candidate_pool) >= n:
            candidates = candidate_pool
            break

    if candidates is None:
        candidates = pool

    sample_n = min(n, len(candidates))
    seed = deterministic_seed_from_id(row_source_id)

    shots = candidates.sample(n=sample_n, random_state=seed)

    keep_cols = [
        "source_id",
        "config",
        "dialect",
        "domain",
        "source_text",
        "target_arabic",
    ]

    return shots[keep_cols].to_dict("records")

def build_few_shot_block(few_shot_examples):
    if not USE_FEW_SHOTS or not few_shot_examples:
        return "No examples available."

    blocks = []

    for i, ex in enumerate(few_shot_examples, start=1):
        ex_config = str(ex.get("config", "")).strip()
        ex_dialect = str(ex.get("dialect", "")).strip()
        ex_domain = str(ex.get("domain", "")).strip()

        meta_parts = []
        if ex_config:
            meta_parts.append(f"config={ex_config}")
        if ex_dialect:
            meta_parts.append(f"dialect={ex_dialect}")
        if ex_domain:
            meta_parts.append(f"domain={ex_domain}")

        meta_line = ", ".join(meta_parts) if meta_parts else "no metadata"

        ex_source = str(ex.get("source_text", "")).strip()
        ex_target = str(ex.get("target_arabic", "")).strip()

        blocks.append(
            f"""Example {i} ({meta_line})
English:
{ex_source}

Arabic:
{ex_target}"""
        )

    return "\n\n".join(blocks)

def make_user_prompt(row):
    context = build_context(row["previous_english_turns"])
    metadata = build_metadata_block(row)
    few_shots = build_few_shot_block(row.get("few_shot_examples", []))

    return f"""Task:
Translate the current English dialogue turn into the target dialectal Arabic variety.

Few-shot training examples:
{few_shots}

Metadata:
{metadata}

Previous English dialogue context:
{context}

Current English turn:
{row["source_text"]}

Rules:
- Preserve the meaning exactly.
- Use the target local dialect, not Modern Standard Arabic unless it is natural in context.
- Follow the dialect/style pattern shown in the few-shot examples when relevant.
- Do not copy the few-shot examples.
- Preserve names, numbers, named entities, and technical terms when appropriate.
- Keep the tone appropriate for the speaker and domain.
- Return only the Arabic translation."""

def row_to_messages(row):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(row)},
        {"role": "assistant", "content": row["target_arabic"]},
    ]

few_shot_pool_cols = [
    "source_id",
    "config",
    "dialect",
    "domain",
    "source_text",
    "target_arabic",
]

train_few_shot_pool = train_df[few_shot_pool_cols].copy()

train_df["few_shot_examples"] = train_df.apply(
    lambda row: select_two_shots_from_train(row, train_few_shot_pool, n=N_FEW_SHOTS),
    axis=1,
)

eval_df["few_shot_examples"] = eval_df.apply(
    lambda row: select_two_shots_from_train(row, train_few_shot_pool, n=N_FEW_SHOTS),
    axis=1,
)

train_df["messages"] = train_df.apply(row_to_messages, axis=1)
eval_df["messages"] = eval_df.apply(row_to_messages, axis=1)

train_dataset = Dataset.from_pandas(
    train_df[
        [
            "messages",
            "source_id",
            "config",
            "domain",
            "dialect",
            "source_text",
            "target_arabic",
        ]
    ],
    preserve_index=False,
)

eval_dataset = Dataset.from_pandas(
    eval_df[
        [
            "messages",
            "source_id",
            "config",
            "domain",
            "dialect",
            "source_text",
            "target_arabic",
        ]
    ],
    preserve_index=False,
)

print(train_dataset)
print(eval_dataset)

print("\nExample few-shot source IDs:")
print([x["source_id"] for x in train_df.iloc[0]["few_shot_examples"]])

print("\nExample messages:")
train_dataset[0]["messages"]

Dataset({
    features: ['messages', 'source_id', 'config', 'domain', 'dialect', 'source_text', 'target_arabic'],
    num_rows: 3108
})
Dataset({
    features: ['messages', 'source_id', 'config', 'domain', 'dialect', 'source_text', 'target_arabic'],
    num_rows: 1118
})

Example few-shot source IDs:
['EG_train_EG_train_79_2', 'EG_train_EG_train_77_2']

Example messages:


[{'content': 'You are a professional machine translation system. Translate the current English dialogue turn into natural dialectal Arabic. Use the provided training examples only as style and dialect guidance. Return only the translation, without explanation.',
  'role': 'system'},
 {'content': "Task:\nTranslate the current English dialogue turn into the target dialectal Arabic variety.\n\nFew-shot training examples:\nExample 1 (config=EG, dialect=Egyptian Arabic (Cairene) Dialect, domain=Agriculture and farming)\nEnglish:\nYes, I believe the original inspection that led to my fine was not done correctly.\n\nArabic:\nأيوه، أنا شايف المعاينة الأولى اللي اتعملت واللي خدت بسببها الغرامة ما كانتش مظبوطة.\n\nExample 2 (config=EG, dialect=Egyptian Arabic (Cairene) Dialect, domain=Agriculture and farming)\nEnglish:\nYes, the project is heavily subsidized. Let me walk you through the proposal and the benefits.\n\nArabic:\nأيوه، المشروع مدعوم بنسبة كبيرة. خليني أشرحلك المقترح وفوايده.\n\nMetad

### Manual SFT/router-corrector format


In [9]:
# ============================================================
# Cell 7 — Manual SFT/router-corrector format used by the LoRA runs
# ============================================================

SYSTEM_MARKER = "### System:"
INSTRUCTION_MARKER = "### Instruction:"
RESPONSE_MARKER = "### Arabic translation:"

def get_message_content(messages, role):
    for m in messages:
        if m.get("role") == role:
            return m.get("content", "")
    return ""

def format_router_prompt(system_text, user_text):
    return (
        f"{SYSTEM_MARKER}\n"
        f"{system_text.strip()}\n\n"
        f"{INSTRUCTION_MARKER}\n"
        f"{user_text.strip()}\n\n"
        f"{RESPONSE_MARKER}\n"
    )

# Backward-compatible alias because older cells used this formatter name.
# This is only a text template function; it does not import or use NTK-Mirror.
format_ntkmirror_prompt = format_router_prompt

def format_sft_text(system_text, user_text, assistant_text=None, add_eos=True):
    text = format_router_prompt(system_text, user_text)

    if assistant_text is not None:
        text += str(assistant_text).strip()

        if add_eos and tokenizer.eos_token is not None:
            text += tokenizer.eos_token

    return text

print("Manual router/SFT template ready.")


Manual router/SFT template ready.


### **Stage one: Qwen base + LoRA adapter**

In [10]:
# ============================================================
# Cell 8 — Load Qwen3-4B Base + frozen LoRA adapter
# ============================================================

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

if LOAD_IN_4BIT:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
else:
    quantization_config = None

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
)

print("Base model loaded.")

model = PeftModel.from_pretrained(
    base_model,
    str(LORA_ADAPTER_PATH),
    is_trainable=False,
)

model.eval()
model.config.use_cache = False

for p in model.parameters():
    p.requires_grad = False

device = next(model.parameters()).device

print("Loaded model:")
print("  base:", MODEL_NAME)
print("  LoRA adapter:", LORA_ADAPTER_PATH)
print("  LoRA best step:", LORA_BEST_STEP)
print("  device:", device)
print("  pad token:", tokenizer.pad_token)
print("  eos token:", tokenizer.eos_token)

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.68k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Base model loaded.
Loaded model:
  base: Qwen/Qwen3-4B-Base
  LoRA adapter: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_alexandria_eg_only_context3_complete2shot_all_group_r16_10epochs/checkpoint-700
  LoRA best step: 700
  device: cuda:0
  pad token: <|endoftext|>
  eos token: <|endoftext|>


In [11]:
# ============================================================
# Cell 9 — Sanity generation with LoRA only, before second-stage corrector
# ============================================================

import torch

def extract_answer(decoded_text):
    if RESPONSE_MARKER in decoded_text:
        answer = decoded_text.split(RESPONSE_MARKER)[-1]
    else:
        answer = decoded_text

    special_tokens = [
        tokenizer.eos_token,
        tokenizer.pad_token,
        "<|endoftext|>",
        "<|im_end|>",
    ]

    for tok in special_tokens:
        if tok:
            answer = answer.replace(tok, "")

    return answer.strip()

def generate_lora_only_from_row(row, max_new_tokens=MAX_NEW_TOKENS):
    user_text = make_user_prompt(row)

    prompt = format_ntkmirror_prompt(
        system_text=SYSTEM_PROMPT,
        user_text=user_text,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)
    answer = extract_answer(decoded)

    return answer, decoded

sample = eval_df.sample(1, random_state=SEED).iloc[0].to_dict()

pred, raw = generate_lora_only_from_row(sample)

print("Config:", sample["config"])
print("Dialect:", sample["dialect"])
print("Domain:", sample["domain"])

print("\nEnglish:")
print(sample["source_text"])

print("\nReference Arabic:")
print(sample["target_arabic"])

print("\nLoRA-only prediction:")
print(pred)


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Config: EG
Dialect: Egyptian Arabic (Cairene) Dialect
Domain: Legal and financial

English:
And can I ask for financial compensation for the damage to my name?

Reference Arabic:
وأقدر أطالب بتعويض مادي عشان الضرر اللي حصل لسمعتي؟

LoRA-only prediction:
وكمان ممكن أطلب تعويض مادي عن الضرر اللي حصل علي اسميا؟


### Building examples for the second-stage LoRA router-corrector


In [12]:
# ============================================================
# Cell 9B — Build LoRA-700 draft cache for second-stage corrector
# Full-train/full-eval draft cache + router diagnostics + balanced mixture
# Reuses existing Stage-1 LoRA drafts and avoids duplicate diagnostic columns
# ============================================================

import json
import math
import re
import shutil
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
from sacrebleu.metrics import BLEU, CHRF

# ------------------------------------------------------------
# Compatibility aliases
# ------------------------------------------------------------

# Some notebook versions use MANUAL_LORA_BEST_STEP; older cells use LORA_BEST_STEP.
if "LORA_BEST_STEP" not in globals():
    if "MANUAL_LORA_BEST_STEP" in globals() and MANUAL_LORA_BEST_STEP is not None:
        LORA_BEST_STEP = MANUAL_LORA_BEST_STEP
    else:
        raise NameError("Neither LORA_BEST_STEP nor MANUAL_LORA_BEST_STEP is defined.")

CORRECTOR_CACHE_DIR = OUTPUT_DIR / "corrector_cache"
CORRECTOR_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Validate config
# ------------------------------------------------------------

assert STAGE2_MODE == "lora_router_corrector", (
    f"Expected STAGE2_MODE='lora_router_corrector', got {STAGE2_MODE}"
)

assert CORRECTOR_TRAIN_CANDIDATE_LIMIT is None, (
    "This experiment must use full train. "
    "Set CORRECTOR_TRAIN_CANDIDATE_LIMIT = None in Cell 3."
)

assert CORRECTOR_EVAL_LIMIT is None, (
    "This experiment must use full eval. "
    "Set CORRECTOR_EVAL_LIMIT = None in Cell 3."
)

assert 0.0 < CORRECTOR_BAD_FRACTION < 1.0, (
    f"CORRECTOR_BAD_FRACTION must be between 0 and 1, got {CORRECTOR_BAD_FRACTION}"
)

assert CORRECTOR_SELECTION_METRIC in {"chrF++", "spBLEU"}, (
    'CORRECTOR_SELECTION_METRIC must be either "chrF++" or "spBLEU"'
)

mix_sum = CORRECTOR_MIX_BAD_ARABIC + CORRECTOR_MIX_GARBAGE + CORRECTOR_MIX_RETAIN
assert abs(mix_sum - 1.0) < 1e-6, (
    "Corrector mixture fractions must sum to 1.0. "
    f"Got {mix_sum}."
)

TRAIN_DRAFT_TAG = "fulltrain"
EVAL_DRAFT_TAG = "fulleval"

metric_tag = CORRECTOR_SELECTION_METRIC.lower().replace("+", "p")
bad_frac_tag = str(CORRECTOR_BAD_FRACTION).replace(".", "p")

TRAIN_DRAFT_PATH = (
    CORRECTOR_CACHE_DIR
    / f"lora_step{LORA_BEST_STEP}_train_drafts_{TRAIN_DRAFT_TAG}.csv"
)

EVAL_DRAFT_PATH = (
    CORRECTOR_CACHE_DIR
    / f"lora_step{LORA_BEST_STEP}_eval_drafts_{EVAL_DRAFT_TAG}.csv"
)

CORRECTOR_TRAIN_PATH = (
    CORRECTOR_CACHE_DIR
    / f"router_corrector_train_fulltrain_mix_badarabic_garbage_retain_"
      f"worst{bad_frac_tag}_{metric_tag}.csv"
)

DIAGNOSTIC_COUNTS_PATH = (
    CORRECTOR_CACHE_DIR
    / f"router_corrector_diagnostic_counts_worst{bad_frac_tag}_{metric_tag}.json"
)

print("Train draft path:", TRAIN_DRAFT_PATH)
print("Eval draft path:", EVAL_DRAFT_PATH)
print("Corrector train path:", CORRECTOR_TRAIN_PATH)
print("Diagnostic counts path:", DIAGNOSTIC_COUNTS_PATH)


# ------------------------------------------------------------
# Helpers: source_id safety
# ------------------------------------------------------------

def ensure_source_id_column(df, split_name):
    df = df.copy()

    if "source_id" not in df.columns:
        df["source_id"] = [f"{split_name}_{i}" for i in range(len(df))]

    df["source_id"] = df["source_id"].astype(str)

    return df


# ------------------------------------------------------------
# Resumable first-stage LoRA draft generation
# ------------------------------------------------------------

def generate_lora_drafts_for_df(input_df, out_path, split_name):
    input_df = ensure_source_id_column(input_df, split_name)
    input_df = input_df.reset_index(drop=True).copy()

    expected_ids = set(input_df["source_id"].astype(str).tolist())

    if out_path.exists():
        existing_df = pd.read_csv(out_path)
        existing_df = ensure_source_id_column(existing_df, split_name)

        # Remove accidental duplicate column labels before any indexing.
        if existing_df.columns.duplicated().any():
            dup_cols = existing_df.columns[existing_df.columns.duplicated()].tolist()
            print(f"{split_name}: dropping duplicated columns from existing cache:")
            print(dup_cols)
            existing_df = existing_df.loc[:, ~existing_df.columns.duplicated()].copy()

        existing_df = existing_df[
            existing_df["source_id"].isin(expected_ids)
        ].copy()

        existing_df = existing_df.drop_duplicates(
            subset=["source_id"],
            keep="first",
        )

        if "lora_draft" not in existing_df.columns:
            existing_df["lora_draft"] = ""

        existing_df["lora_draft"] = existing_df["lora_draft"].fillna("").astype(str)

        done_df = existing_df[
            existing_df["lora_draft"].str.strip().str.len() > 0
        ].copy()

        rows = done_df.to_dict("records")
        done_ids = set(done_df["source_id"].astype(str).tolist())

        print(f"Resuming {split_name} drafts: {len(done_ids)} / {len(input_df)} already done.")

    else:
        rows = []
        done_ids = set()
        print(f"No existing {split_name} draft file. Starting from scratch.")

    generated_since_save = 0

    missing_df = input_df[
        ~input_df["source_id"].isin(done_ids)
    ].reset_index(drop=True)

    for _, row in tqdm(
        missing_df.iterrows(),
        total=len(missing_df),
        desc=f"Generate LoRA-{LORA_BEST_STEP} drafts for {split_name}",
    ):
        row_dict = row.to_dict()
        source_id = str(row_dict["source_id"])

        try:
            draft, _ = generate_lora_only_from_row(row_dict)
        except Exception as e:
            draft = ""
            print("Draft generation failed:", source_id, repr(e))

        few_shot_block = build_few_shot_block(row_dict.get("few_shot_examples", []))
        metadata_block = build_metadata_block(row_dict)
        previous_context_block = build_context(row_dict.get("previous_english_turns", []))

        rows.append({
            "source_id": source_id,
            "split": row_dict.get("split", split_name),
            "config": row_dict.get("config", ""),
            "conversation_id": row_dict.get("conversation_id", ""),
            "turn_id": row_dict.get("turn_id", ""),
            "country": row_dict.get("country", ""),
            "dialect": row_dict.get("dialect", ""),
            "domain": row_dict.get("domain", ""),
            "speaker": row_dict.get("speaker", ""),
            "gender_direction": row_dict.get("gender_direction", ""),
            "source_text": row_dict.get("source_text", ""),
            "target_arabic": row_dict.get("target_arabic", ""),
            "few_shot_block": few_shot_block,
            "metadata_block": metadata_block,
            "previous_context_block": previous_context_block,
            "lora_draft": draft,
        })

        generated_since_save += 1

        if generated_since_save >= DRAFT_SAVE_EVERY:
            tmp_df = pd.DataFrame(rows)
            tmp_df["source_id"] = tmp_df["source_id"].astype(str)
            tmp_df = tmp_df.drop_duplicates(subset=["source_id"], keep="first")
            tmp_df.to_csv(out_path, index=False, encoding="utf-8-sig")
            generated_since_save = 0

    out_df = pd.DataFrame(rows)

    if len(out_df) == 0:
        raise RuntimeError(f"No drafts available for {split_name}.")

    if out_df.columns.duplicated().any():
        dup_cols = out_df.columns[out_df.columns.duplicated()].tolist()
        print(f"{split_name}: dropping duplicated columns from out_df:")
        print(dup_cols)
        out_df = out_df.loc[:, ~out_df.columns.duplicated()].copy()

    out_df["source_id"] = out_df["source_id"].astype(str)
    out_df = out_df.drop_duplicates(subset=["source_id"], keep="first")

    order_df = input_df[["source_id"]].copy()
    order_df["source_id"] = order_df["source_id"].astype(str)

    out_df = order_df.merge(out_df, on="source_id", how="left")

    if out_df.columns.duplicated().any():
        dup_cols = out_df.columns[out_df.columns.duplicated()].tolist()
        raise RuntimeError(f"{split_name}: duplicate columns after merge: {dup_cols}")

    out_df.to_csv(out_path, index=False, encoding="utf-8-sig")

    missing_drafts = out_df["lora_draft"].isna().sum()
    empty_drafts = (
        out_df["lora_draft"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.len()
        == 0
    ).sum()

    print(f"Saved {split_name} drafts:", out_path)
    print("Rows:", len(out_df))
    print("Missing drafts:", int(missing_drafts))
    print("Empty drafts:", int(empty_drafts))

    return out_df


# ------------------------------------------------------------
# Use full train and full eval
# ------------------------------------------------------------

train_for_drafts_df = ensure_source_id_column(train_df.reset_index(drop=True), "train")
eval_for_drafts_df = ensure_source_id_column(eval_df.reset_index(drop=True), "eval")

print("Train draft examples:", len(train_for_drafts_df))
print("Eval draft examples:", len(eval_for_drafts_df))
print("Full train draft mode confirmed:", len(train_for_drafts_df), "examples")
print("Full eval draft mode confirmed:", len(eval_for_drafts_df), "examples")


# ------------------------------------------------------------
# Reuse already-created Stage-1 LoRA draft caches if available
# ------------------------------------------------------------

def find_existing_stage1_draft_cache(filename):
    """
    Find an already-created Stage-1 LoRA draft cache from previous runs.
    Prefer old NTK/corrector_cache folders.
    Exclude the current destination file.
    """
    candidates = []

    for root in [RUNS_DIR, PROJECT_DIR]:
        root = Path(root)

        if not root.exists():
            continue

        for p in root.rglob(filename):
            p = Path(p)

            # Do not select the current new experiment destination.
            if p.resolve() == (CORRECTOR_CACHE_DIR / filename).resolve():
                continue

            path_text = str(p).lower()

            score = 0

            if "corrector_cache" in path_text:
                score += 50

            if "ntk" in path_text or "ntkmirror" in path_text:
                score += 50

            if f"lora_step{LORA_BEST_STEP}" in path_text:
                score += 25

            candidates.append((score, p))

    if not candidates:
        return None

    candidates = sorted(candidates, key=lambda x: x[0], reverse=True)

    return candidates[0][1]


def copy_existing_stage1_cache_if_valid(filename, dst_path, expected_rows):
    """
    Copy previous Stage-1 draft cache into the current experiment cache folder.
    If dst already exists and is complete, do nothing.
    """
    dst_path = Path(dst_path)

    if dst_path.exists():
        try:
            dst_df = pd.read_csv(dst_path)

            # Remove accidental duplicate columns before validation.
            if dst_df.columns.duplicated().any():
                dup_cols = dst_df.columns[dst_df.columns.duplicated()].tolist()
                print("Current cache has duplicate columns; will keep cache but clean later:")
                print(dup_cols)

            if len(dst_df) >= expected_rows and "lora_draft" in dst_df.columns:
                print(f"Using existing current cache: {dst_path}")
                print("Rows:", len(dst_df))
                return True

            print(f"Current cache exists but is incomplete: {dst_path}")
            print("Rows:", len(dst_df), "Expected:", expected_rows)

        except Exception as e:
            print("Could not read current cache:", dst_path)
            print("Reason:", repr(e))

    old_path = find_existing_stage1_draft_cache(filename)

    if old_path is None:
        print(f"No previous Stage-1 draft cache found for: {filename}")
        return False

    old_df = pd.read_csv(old_path)

    if len(old_df) < expected_rows:
        print(f"Found old cache but it is incomplete: {old_path}")
        print("Rows:", len(old_df), "Expected:", expected_rows)
        return False

    if "lora_draft" not in old_df.columns:
        print(f"Found old cache but it has no lora_draft column: {old_path}")
        return False

    dst_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(old_path, dst_path)

    print("Copied previous Stage-1 draft cache:")
    print("  from:", old_path)
    print("  to:  ", dst_path)
    print("  rows:", len(old_df))

    return True


train_cache_filename = f"lora_step{LORA_BEST_STEP}_train_drafts_{TRAIN_DRAFT_TAG}.csv"
eval_cache_filename = f"lora_step{LORA_BEST_STEP}_eval_drafts_{EVAL_DRAFT_TAG}.csv"

copy_existing_stage1_cache_if_valid(
    filename=train_cache_filename,
    dst_path=TRAIN_DRAFT_PATH,
    expected_rows=len(train_for_drafts_df),
)

copy_existing_stage1_cache_if_valid(
    filename=eval_cache_filename,
    dst_path=EVAL_DRAFT_PATH,
    expected_rows=len(eval_for_drafts_df),
)


# ------------------------------------------------------------
# Generate or resume LoRA-700 drafts
# ------------------------------------------------------------
# If the previous cache was copied successfully, this will simply load/resume
# from the copied CSV and generate zero new drafts.

train_drafts_df = generate_lora_drafts_for_df(
    train_for_drafts_df,
    TRAIN_DRAFT_PATH,
    split_name="train",
)

eval_drafts_df = generate_lora_drafts_for_df(
    eval_for_drafts_df,
    EVAL_DRAFT_PATH,
    split_name="eval",
)


# ------------------------------------------------------------
# Sentence metrics + script / garbage diagnostics
# ------------------------------------------------------------

# effective_order=True avoids thousands of warnings during sentence-level BLEU.
chrfpp_metric = CHRF(word_order=2)
spbleu_metric = BLEU(tokenize="flores200", effective_order=True)


def sentence_chrfpp(pred, ref):
    pred = "" if pd.isna(pred) else str(pred)
    ref = "" if pd.isna(ref) else str(ref)

    try:
        return float(chrfpp_metric.sentence_score(pred, [ref]).score)
    except Exception:
        return 0.0


def sentence_spbleu(pred, ref):
    pred = "" if pd.isna(pred) else str(pred)
    ref = "" if pd.isna(ref) else str(ref)

    try:
        return float(spbleu_metric.sentence_score(pred, [ref]).score)
    except Exception:
        return 0.0


ARABIC_RE = re.compile(r"[\u0600-\u06FF\u0750-\u077F\u08A0-\u08FF\uFB50-\uFDFF\uFE70-\uFEFF]")
HAN_RE = re.compile(r"[\u3400-\u4DBF\u4E00-\u9FFF\uF900-\uFAFF]")
LATIN_RE = re.compile(r"[A-Za-z]")


def script_diagnostics(text):
    text = "" if pd.isna(text) else str(text)
    stripped = text.strip()
    chars = [ch for ch in stripped if not ch.isspace()]
    total = len(chars)

    arabic_count = len(ARABIC_RE.findall(stripped))
    han_count = len(HAN_RE.findall(stripped))
    latin_count = len(LATIN_RE.findall(stripped))

    arabic_ratio = arabic_count / total if total else 0.0
    han_ratio = han_count / total if total else 0.0
    latin_ratio = latin_count / total if total else 0.0

    has_chinese = han_count > 0
    empty_draft = len(stripped) == 0

    mostly_non_arabic = (
        (not empty_draft)
        and arabic_ratio < ARABIC_RATIO_MIN_FOR_ARABIC
        and (latin_ratio + han_ratio) >= NON_ARABIC_RATIO_FOR_GARBAGE
    )

    garbage_draft = bool(empty_draft or has_chinese or mostly_non_arabic)

    return {
        "arabic_ratio": float(arabic_ratio),
        "han_ratio": float(han_ratio),
        "latin_ratio": float(latin_ratio),
        "has_chinese": bool(has_chinese),
        "mostly_non_arabic": bool(mostly_non_arabic),
        "empty_draft": bool(empty_draft),
        "garbage_draft": bool(garbage_draft),
    }


def add_draft_diagnostics(df):
    df = df.copy()

    # --------------------------------------------------------
    # If we reused old caches, they may already contain these
    # diagnostic columns. Drop them before recomputing.
    # Otherwise pd.concat creates duplicate columns and later
    # pandas fails with:
    # ValueError: cannot reindex on an axis with duplicate labels
    # --------------------------------------------------------

    diagnostic_cols = [
        "draft_chrF++",
        "draft_spBLEU",
        "arabic_ratio",
        "han_ratio",
        "latin_ratio",
        "has_chinese",
        "mostly_non_arabic",
        "empty_draft",
        "garbage_draft",
        "draft_group",
    ]

    # Remove duplicated column labels if they already exist.
    if df.columns.duplicated().any():
        dup_cols = df.columns[df.columns.duplicated()].tolist()
        print("Dropping duplicated column labels before diagnostics:")
        print(dup_cols)
        df = df.loc[:, ~df.columns.duplicated()].copy()

    # Drop old diagnostic columns before recomputing.
    existing_diag_cols = [c for c in diagnostic_cols if c in df.columns]

    if existing_diag_cols:
        print("Dropping old diagnostic columns before recomputing:")
        print(existing_diag_cols)
        df = df.drop(columns=existing_diag_cols)

    # Drop suffixed diagnostic columns like arabic_ratio.1 from failed reruns.
    suffix_diag_cols = []
    for c in df.columns:
        c_str = str(c)
        base = c_str.split(".")[0]
        if base in diagnostic_cols:
            suffix_diag_cols.append(c)

    if suffix_diag_cols:
        print("Dropping suffixed old diagnostic columns:")
        print(suffix_diag_cols)
        df = df.drop(columns=suffix_diag_cols)

    # Final duplicate-column cleanup before recomputation.
    if df.columns.duplicated().any():
        dup_cols = df.columns[df.columns.duplicated()].tolist()
        raise RuntimeError(f"Duplicate columns remain before diagnostics: {dup_cols}")

    if "lora_draft" not in df.columns:
        raise RuntimeError("Missing lora_draft column before diagnostics.")

    if "target_arabic" not in df.columns:
        raise RuntimeError("Missing target_arabic column before diagnostics.")

    df["lora_draft"] = df["lora_draft"].fillna("").astype(str)
    df["target_arabic"] = df["target_arabic"].fillna("").astype(str)

    df["draft_chrF++"] = df.apply(
        lambda r: sentence_chrfpp(r["lora_draft"], r["target_arabic"]),
        axis=1,
    )

    df["draft_spBLEU"] = df.apply(
        lambda r: sentence_spbleu(r["lora_draft"], r["target_arabic"]),
        axis=1,
    )

    diag_df = pd.DataFrame(
        [script_diagnostics(x) for x in df["lora_draft"].tolist()]
    )

    df = pd.concat(
        [df.reset_index(drop=True), diag_df.reset_index(drop=True)],
        axis=1,
    )

    # Final hard check.
    if df.columns.duplicated().any():
        dup_cols = df.columns[df.columns.duplicated()].tolist()
        raise RuntimeError(f"Duplicate columns remain after diagnostics: {dup_cols}")

    return df


train_drafts_df = add_draft_diagnostics(train_drafts_df)
eval_drafts_df = add_draft_diagnostics(eval_drafts_df)


# ------------------------------------------------------------
# Hard safety cleanup before group selection
# ------------------------------------------------------------

def assert_no_duplicate_columns(df, name):
    dup_cols = df.columns[df.columns.duplicated()].tolist()

    if dup_cols:
        print(f"{name} duplicate columns:", dup_cols)
        raise RuntimeError(f"{name} still has duplicate columns.")

    print(f"✅ {name}: no duplicate columns")


assert_no_duplicate_columns(train_drafts_df, "train_drafts_df")
assert_no_duplicate_columns(eval_drafts_df, "eval_drafts_df")

# Force garbage_draft to be a clean boolean Series.
train_drafts_df["garbage_draft"] = train_drafts_df["garbage_draft"].astype(bool)
eval_drafts_df["garbage_draft"] = eval_drafts_df["garbage_draft"].astype(bool)

selection_score_col = f"draft_{CORRECTOR_SELECTION_METRIC}"

if selection_score_col not in train_drafts_df.columns:
    raise RuntimeError(
        f"Missing selection score column: {selection_score_col}. "
        f"Available columns: {train_drafts_df.columns.tolist()}"
    )


# ------------------------------------------------------------
# Build router groups
# ------------------------------------------------------------

non_garbage_train_df = train_drafts_df[
    ~train_drafts_df["garbage_draft"]
].copy()

garbage_df = train_drafts_df[
    train_drafts_df["garbage_draft"]
].copy()

if len(non_garbage_train_df) > 0:
    bad_cutoff = float(
        non_garbage_train_df[selection_score_col].quantile(CORRECTOR_BAD_FRACTION)
    )
else:
    bad_cutoff = 0.0

bad_arabic_df = non_garbage_train_df[
    non_garbage_train_df[selection_score_col] <= bad_cutoff
].copy()

retain_df = non_garbage_train_df[
    non_garbage_train_df[selection_score_col] > bad_cutoff
].copy()

bad_arabic_df["draft_group"] = "bad_arabic"
garbage_df["draft_group"] = "garbage"
retain_df["draft_group"] = "retain"

# Add group labels to full draft files too.
train_drafts_df["draft_group"] = "unassigned"
train_drafts_df.loc[bad_arabic_df.index, "draft_group"] = "bad_arabic"
train_drafts_df.loc[garbage_df.index, "draft_group"] = "garbage"
train_drafts_df.loc[retain_df.index, "draft_group"] = "retain"

eval_drafts_df["draft_group"] = np.where(
    eval_drafts_df["garbage_draft"],
    "garbage",
    "non_garbage",
)

# Save full draft caches again, now with clean diagnostics.
train_drafts_df.to_csv(TRAIN_DRAFT_PATH, index=False, encoding="utf-8-sig")
eval_drafts_df.to_csv(EVAL_DRAFT_PATH, index=False, encoding="utf-8-sig")


# ------------------------------------------------------------
# Build balanced router-corrector mixture from full-train draft cache
# ------------------------------------------------------------

def sample_group(df, requested_n, group_name, seed_offset=0):
    requested_n = int(requested_n)

    if requested_n <= 0:
        out = df.iloc[0:0].copy()
        out["sampled_with_replacement"] = False
        out["requested_group_n"] = requested_n
        return out, False, requested_n

    if len(df) == 0:
        print(
            f"WARNING: requested {requested_n} rows for group '{group_name}', "
            "but the group is empty. Reducing this group to 0."
        )
        out = df.iloc[0:0].copy()
        out["sampled_with_replacement"] = False
        out["requested_group_n"] = requested_n
        return out, False, 0

    replace = len(df) < requested_n

    out = df.sample(
        n=requested_n,
        replace=replace,
        random_state=SEED + seed_offset,
    ).copy()

    out["sampled_with_replacement"] = bool(replace)
    out["requested_group_n"] = requested_n

    if replace:
        print(
            f"Group '{group_name}' has only {len(df)} rows but requested {requested_n}; "
            "sampling with replacement."
        )

    return out, replace, requested_n


requested_total = (
    len(train_drafts_df)
    if CORRECTOR_TRAIN_TARGET_TOTAL is None
    else int(CORRECTOR_TRAIN_TARGET_TOTAL)
)

requested_bad = int(round(requested_total * CORRECTOR_MIX_BAD_ARABIC))
requested_garbage = int(round(requested_total * CORRECTOR_MIX_GARBAGE))
requested_retain = requested_total - requested_bad - requested_garbage

bad_sample_df, bad_repl, actual_bad_request = sample_group(
    bad_arabic_df,
    requested_bad,
    "bad_arabic",
    seed_offset=11,
)

garbage_sample_df, garbage_repl, actual_garbage_request = sample_group(
    garbage_df,
    requested_garbage,
    "garbage",
    seed_offset=22,
)

retain_sample_df, retain_repl, actual_retain_request = sample_group(
    retain_df,
    requested_retain,
    "retain",
    seed_offset=33,
)

corrector_train_df = pd.concat(
    [bad_sample_df, garbage_sample_df, retain_sample_df],
    axis=0,
    ignore_index=True,
)

if len(corrector_train_df) == 0:
    raise RuntimeError(
        "Router-corrector train set is empty. Check draft cache generation and diagnostics."
    )

# Final duplicate check before saving train file.
if corrector_train_df.columns.duplicated().any():
    dup_cols = corrector_train_df.columns[corrector_train_df.columns.duplicated()].tolist()
    raise RuntimeError(f"corrector_train_df has duplicate columns: {dup_cols}")

corrector_train_df = corrector_train_df.sample(
    frac=1.0,
    random_state=SEED,
).reset_index(drop=True)

corrector_train_df["selected_for_corrector"] = True
corrector_train_df["selection_metric"] = CORRECTOR_SELECTION_METRIC
corrector_train_df["selection_fraction"] = CORRECTOR_BAD_FRACTION
corrector_train_df["bad_cutoff"] = bad_cutoff
corrector_train_df["router_mix_bad_arabic"] = CORRECTOR_MIX_BAD_ARABIC
corrector_train_df["router_mix_garbage"] = CORRECTOR_MIX_GARBAGE
corrector_train_df["router_mix_retain"] = CORRECTOR_MIX_RETAIN
corrector_train_df["selection_rank"] = np.arange(1, len(corrector_train_df) + 1)
corrector_train_df["sample_uid"] = [f"router_train_{i:06d}" for i in range(len(corrector_train_df))]

corrector_eval_df = eval_drafts_df.copy().reset_index(drop=True)

corrector_train_df.to_csv(CORRECTOR_TRAIN_PATH, index=False, encoding="utf-8-sig")


# ------------------------------------------------------------
# Diagnostic counts
# ------------------------------------------------------------

counts = {
    "full_train_rows": int(len(train_drafts_df)),
    "full_eval_rows": int(len(eval_drafts_df)),
    "selection_metric": CORRECTOR_SELECTION_METRIC,
    "bad_fraction": float(CORRECTOR_BAD_FRACTION),
    "bad_cutoff": float(bad_cutoff),
    "available_bad_arabic": int(len(bad_arabic_df)),
    "available_garbage": int(len(garbage_df)),
    "available_retain": int(len(retain_df)),
    "requested_total": int(requested_total),
    "requested_bad_arabic": int(requested_bad),
    "requested_garbage": int(requested_garbage),
    "requested_retain": int(requested_retain),
    "actual_train_rows": int(len(corrector_train_df)),
    "actual_bad_arabic_rows": int((corrector_train_df["draft_group"] == "bad_arabic").sum()),
    "actual_garbage_rows": int((corrector_train_df["draft_group"] == "garbage").sum()),
    "actual_retain_rows": int((corrector_train_df["draft_group"] == "retain").sum()),
    "bad_arabic_sampled_with_replacement": bool(bad_repl),
    "garbage_sampled_with_replacement": bool(garbage_repl),
    "retain_sampled_with_replacement": bool(retain_repl),
    "train_empty_drafts": int(train_drafts_df["empty_draft"].sum()),
    "train_has_chinese": int(train_drafts_df["has_chinese"].sum()),
    "train_mostly_non_arabic": int(train_drafts_df["mostly_non_arabic"].sum()),
    "train_garbage_drafts": int(train_drafts_df["garbage_draft"].sum()),
    "eval_empty_drafts": int(eval_drafts_df["empty_draft"].sum()),
    "eval_has_chinese": int(eval_drafts_df["has_chinese"].sum()),
    "eval_mostly_non_arabic": int(eval_drafts_df["mostly_non_arabic"].sum()),
    "eval_garbage_drafts": int(eval_drafts_df["garbage_draft"].sum()),
}

DIAGNOSTIC_COUNTS_PATH.write_text(
    json.dumps(counts, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("\nFull-train draft diagnostics and router mixture:")
print(json.dumps(counts, indent=2, ensure_ascii=False))

assert len(train_drafts_df) == len(train_df), (
    f"Expected full train size {len(train_df)}, got {len(train_drafts_df)}"
)

assert len(corrector_eval_df) == len(eval_df), (
    f"Expected full eval size {len(eval_df)}, got {len(corrector_eval_df)}"
)

assert_no_duplicate_columns(train_drafts_df, "final train_drafts_df")
assert_no_duplicate_columns(eval_drafts_df, "final eval_drafts_df")
assert_no_duplicate_columns(corrector_train_df, "final corrector_train_df")

print("\nFull train and full eval coverage confirmed.")
print("Saved train draft cache:", TRAIN_DRAFT_PATH)
print("Saved eval draft cache:", EVAL_DRAFT_PATH)
print("Saved corrector train file:", CORRECTOR_TRAIN_PATH)
print("Saved diagnostic counts:", DIAGNOSTIC_COUNTS_PATH)

print("\nTrain draft score stats:")
display(
    train_drafts_df[
        ["draft_chrF++", "draft_spBLEU", "arabic_ratio", "han_ratio", "latin_ratio"]
    ].describe().T
)

print("\nCorrector train group counts:")
display(corrector_train_df["draft_group"].value_counts(dropna=False).to_frame("count"))

print("\nCorrector train score stats by group:")
display(
    corrector_train_df.groupby("draft_group")[["draft_chrF++", "draft_spBLEU"]]
    .describe()
)

print("\nCorrector train preview:")
display(
    corrector_train_df[
        [
            "sample_uid",
            "source_id",
            "draft_group",
            "source_text",
            "lora_draft",
            "target_arabic",
            "draft_chrF++",
            "draft_spBLEU",
            "arabic_ratio",
            "han_ratio",
            "latin_ratio",
            "garbage_draft",
        ]
    ].head(10)
)

Train draft path: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_lora_step700_alexandria_eg_only_context3_complete2shot_LORA_ROUTER_CORRECTOR_draft2gold_fulltrain_mix_badarabic_garbage_retain_worst0p3_chrfpp_fulleval_r8_alpha16_drop0p05_lr5e-05_3epochs/corrector_cache/lora_step700_train_drafts_fulltrain.csv
Eval draft path: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_lora_step700_alexandria_eg_only_context3_complete2shot_LORA_ROUTER_CORRECTOR_draft2gold_fulltrain_mix_badarabic_garbage_retain_worst0p3_chrfpp_fulleval_r8_alpha16_drop0p05_lr5e-05_3epochs/corrector_cache/lora_step700_eval_drafts_fulleval.csv
Corrector train path: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_lora_step700_alexandria_eg_only_context3_complete2shot_LORA_ROUTER_CORRECTOR_draft2gold_fulltrain_mix_badarabic_garbage_retain_worst0p3_chrfpp_fulleval_r8_alpha16_drop0p05_lr5e-05_3epochs/corrector_cache/router_corrector_train_fulltrain_mix_badarabic_garbage_retain_worst0p3_chrfp

Generate LoRA-700 drafts for train: 0it [00:00, ?it/s]

Saved train drafts: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_lora_step700_alexandria_eg_only_context3_complete2shot_LORA_ROUTER_CORRECTOR_draft2gold_fulltrain_mix_badarabic_garbage_retain_worst0p3_chrfpp_fulleval_r8_alpha16_drop0p05_lr5e-05_3epochs/corrector_cache/lora_step700_train_drafts_fulltrain.csv
Rows: 3108
Missing drafts: 0
Empty drafts: 0
Resuming eval drafts: 1118 / 1118 already done.


Generate LoRA-700 drafts for eval: 0it [00:00, ?it/s]

Saved eval drafts: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_lora_step700_alexandria_eg_only_context3_complete2shot_LORA_ROUTER_CORRECTOR_draft2gold_fulltrain_mix_badarabic_garbage_retain_worst0p3_chrfpp_fulleval_r8_alpha16_drop0p05_lr5e-05_3epochs/corrector_cache/lora_step700_eval_drafts_fulleval.csv
Rows: 1118
Missing drafts: 0
Empty drafts: 0
Dropping old diagnostic columns before recomputing:
['draft_chrF++', 'draft_spBLEU', 'arabic_ratio', 'han_ratio', 'latin_ratio', 'has_chinese', 'mostly_non_arabic', 'empty_draft', 'garbage_draft', 'draft_group']
Dropping old diagnostic columns before recomputing:
['draft_chrF++', 'draft_spBLEU', 'arabic_ratio', 'han_ratio', 'latin_ratio', 'has_chinese', 'mostly_non_arabic', 'empty_draft', 'garbage_draft', 'draft_group']
✅ train_drafts_df: no duplicate columns
✅ eval_drafts_df: no duplicate columns
Group 'bad_arabic' has only 928 rows but requested 1554; sampling with replacement.
Group 'garbage' has only 14 rows but requested 7

...

,count
draft_group,
bad_arabic,1554
retain,777
garbage,777



Corrector train score stats by group:


draft_chrF++                                              \
                   count       mean        std        min        25%   
draft_group                                                            
bad_arabic        1554.0  28.288804   6.925003   1.205546  24.219716   
garbage            777.0  23.515910  19.849106   0.000000   3.847736   
retain             777.0  56.648473  13.868920  37.928827  45.533112   

                                              draft_spBLEU             \
                   50%        75%         max        count       mean   
draft_group                                                             
bad_arabic   29.504472  33.731604   37.895053       1554.0  12.669199   
garbage      15.447399  38.416940   55.423025        777.0  14.723455   
retain       53.612650  65.019235  100.000000        777.0  42.471302   

                                                                               
                   std       min        25%        50%        75%         max  
draft_group                                                                    
bad_arabic    6.665358  0.000000   7.930495  11.601529  17.466255   53.107253  
garbage      14.373747  0.000000   1.376612  10.858944  26.573593   40.660574  
retain       18.326269  8.295194  29.071537  39.451227  53.394988  100.000000


Corrector train preview:


,sample_uid,source_id,draft_group,source_text,lora_draft,target_arabic,draft_chrF++,draft_spBLEU,arabic_ratio,han_ratio,latin_ratio,garbage_draft
0,router_train_000000,EG_train_EG_train_931_1,bad_arabic,We absolutely need a 'buddy system'. Assigning...,"احنا فعلا محتاجين نظام ""الرجل الواحد"" نعين حد ...",إحنا فعلا محتاجين نعمل “نظام اخوة” كدة اللي هو...,30.989580,15.373671,0.972603,0.000000,0.000000,False
1,router_train_000001,EG_train_EG_train_10_2,bad_arabic,Alright. Let's start the treatment on those tw...,تمام. خلينا نبدأ العلاج على التينين ديه حالا و...,تمام، خلينا نبدأ العلاج عالاتنين دول على طول، ...,36.942279,21.084455,0.966102,0.000000,0.000000,False
2,router_train_000002,EG_train_EG_train_629_2,retain,"He saved for two years! In those days, a car w...",هو جمع لسنتين! في وقتها العربية كانت حاجة كبير...,حوش لمدة سنتين! اياميها، العربية كانت حاجة كبي...,38.733631,23.989326,0.811594,0.000000,0.144928,False
3,router_train_000003,EG_train_EG_train_187_2,garbage,"Please bend your knees slightly, like you're s...",لو سمحت ركبي قدميك شوية كأنك جالس في كرسي بس م...,اتنوا ركبكم شوية، كإنكم قاعدين على كرسي بس من ...,35.071291,19.080503,0.946237,0.021505,0.000000,True
4,router_train_000004,EG_train_EG_train_949_3,bad_arabic,You're welcome. You get used to it!,العفو. هتعودي عادتك!,العفو. هتتعود على الموضوع!,37.484129,33.031643,0.888889,0.000000,0.000000,False
5,router_train_000005,EG_train_EG_train_518_0,retain,"Good morning, Sarah. I'm looking at the latest...",صباح الخير يا سارة. أنا ببص على أخر تقرير لمسا...,صباح الخير يا سارة، انا بدور على اجدد تقرير مس...,39.839187,26.225217,0.977273,0.000000,0.000000,False
6,router_train_000006,EG_train_EG_train_434_0,garbage,Did you see the email about the new fees? They...,شفتي الإيميل عن المصاريف الجديدة؟ ازدادت تاني!...,شفتي الإيميل بتاع المصاريف الجديدة؟ زودوها تان...,54.581071,39.645133,0.920000,0.040000,0.000000,True
7,router_train_000007,EG_train_EG_train_455_2,bad_arabic,Was a formal risk assessment conducted and doc...,كان في تقييم خطر رسمي وتم سجله وقت اتخاذ قرار ...,طيب وعملتي التقرير الرسمي للمخاطر عشان التأجيل...,16.500123,1.901176,1.000000,0.000000,0.000000,False
8,router_train_000008,EG_train_EG_train_612_0,retain,"Sorry to bother you, could you please tell me ...",بعتذر أضايقك، ممكن تقوليلي فين أقرب صيدلية؟,اسف لإزعجك، ممكن تقوليلي فين أقرب صيدلية؟,73.775196,65.536096,1.000000,0.000000,0.000000,False
9,router_train_000009,EG_train_EG_train_20_2,bad_arabic,"Okay, for how long should I run the system eac...",تمام، المفروض أعمل النظام ده قدام قد إيه؟,تمام، أد ايه من الوقت المفروض استعمل النظام ده...,36.521201,28.648683,1.000000,0.000000,0.000000,False


### **Build router-corrector LoRA prompts**


In [13]:
# ============================================================
# Cell 10 — Build router-corrector LoRA training prompts
# Source/context + first-stage LoRA draft → gold target_arabic
# Labels mask the prompt and train only on the target completion
# ============================================================

import pandas as pd
import torch
from torch.utils.data import Dataset

CORRECTOR_SYSTEM_PROMPT = (
    "You are a professional Egyptian Arabic machine translation router-corrector. "
    "You receive the original English dialogue turn, context, metadata, examples, "
    "and a draft Arabic translation from a first-stage LoRA model. "
    "Your job is to decide whether to correct the draft, ignore it, or keep it close. "
    "Return only the final Egyptian Arabic translation, without explanation."
)


def safe_cell_value(row, key, default=""):
    value = row.get(key, default)

    if value is None:
        return default

    try:
        if pd.isna(value):
            return default
    except Exception:
        pass

    return str(value)


def bool_cell_value(row, key, default=False):
    value = row.get(key, default)

    if isinstance(value, bool):
        return value

    if isinstance(value, (int, float)):
        return bool(value)

    value = str(value).strip().lower()

    if value in {"true", "1", "yes", "y"}:
        return True

    if value in {"false", "0", "no", "n", ""}:
        return False

    return bool(default)


def make_corrector_user_prompt(row):
    few_shot_block = safe_cell_value(row, "few_shot_block", "No examples available.")
    metadata_block = safe_cell_value(row, "metadata_block", "No metadata.")
    previous_context_block = safe_cell_value(row, "previous_context_block", "No previous context.")
    source_text = safe_cell_value(row, "source_text", "")
    lora_draft = safe_cell_value(row, "lora_draft", "")

    draft_group = safe_cell_value(row, "draft_group", "unknown")
    draft_chrfpp = safe_cell_value(row, "draft_chrF++", "")
    draft_spbleu = safe_cell_value(row, "draft_spBLEU", "")
    arabic_ratio = safe_cell_value(row, "arabic_ratio", "")
    han_ratio = safe_cell_value(row, "han_ratio", "")
    latin_ratio = safe_cell_value(row, "latin_ratio", "")
    has_chinese = safe_cell_value(row, "has_chinese", "")
    mostly_non_arabic = safe_cell_value(row, "mostly_non_arabic", "")
    empty_draft = safe_cell_value(row, "empty_draft", "")
    garbage_draft = safe_cell_value(row, "garbage_draft", "")

    return f"""Task:
Produce the final Egyptian Arabic translation for the current English dialogue turn.
You are given the original source/context and a first-stage LoRA draft.
This is a router-corrector task, not a blind draft editor.

Few-shot training examples:
{few_shot_block}

Metadata:
{metadata_block}

Previous English dialogue context:
{previous_context_block}

Current English turn:
{source_text}

First-stage LoRA draft translation:
{lora_draft}

Draft diagnostics:
- draft_group: {draft_group}
- draft_chrF++: {draft_chrfpp}
- draft_spBLEU: {draft_spbleu}
- arabic_ratio: {arabic_ratio}
- han_ratio: {han_ratio}
- latin_ratio: {latin_ratio}
- has_chinese: {has_chinese}
- mostly_non_arabic: {mostly_non_arabic}
- empty_draft: {empty_draft}
- garbage_draft: {garbage_draft}

Router-corrector rules:
1. If the draft is useful Arabic but weak, correct or rewrite it into natural Egyptian Arabic.
2. If the draft is garbage, Chinese, wrong-language, empty, unrelated, or mostly non-Arabic, ignore the draft and retranslate from the English source/context.
3. If the draft is already good, keep it close and do not over-edit.
4. Preserve the exact meaning, names, numbers, named entities, and technical terms when appropriate.
5. Use natural Egyptian Arabic/local dialect style, not stiff MSA unless it is natural in context.
6. Do not explain your decision and do not mention the draft diagnostics.
7. Return only the final Egyptian Arabic translation."""


def make_corrector_prompt(row):
    return format_ntkmirror_prompt(
        system_text=CORRECTOR_SYSTEM_PROMPT,
        user_text=make_corrector_user_prompt(row),
    )


def make_corrector_completion(row):
    completion = safe_cell_value(row, "target_arabic", "").strip()

    if tokenizer.eos_token is not None:
        completion += tokenizer.eos_token

    return completion


class CorrectorSFTDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=MAX_SEQ_LENGTH):
        self.df = df.reset_index(drop=True).copy()
        self.tokenizer = tokenizer
        self.max_length = int(max_length)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx].to_dict()

        prompt = make_corrector_prompt(row)
        completion = make_corrector_completion(row)

        prompt_ids = self.tokenizer(
            prompt,
            add_special_tokens=False,
            truncation=False,
        )["input_ids"]

        completion_ids = self.tokenizer(
            completion,
            add_special_tokens=False,
            truncation=False,
        )["input_ids"]

        if len(completion_ids) == 0:
            completion_ids = [self.tokenizer.eos_token_id]

        # Keep at least the target completion and the tail of the prompt.
        if len(completion_ids) >= self.max_length:
            completion_ids = completion_ids[: self.max_length - 1]
            if self.tokenizer.eos_token_id is not None:
                completion_ids.append(self.tokenizer.eos_token_id)

        max_prompt_len = self.max_length - len(completion_ids)
        if max_prompt_len <= 0:
            max_prompt_len = 1

        if len(prompt_ids) > max_prompt_len:
            prompt_ids = prompt_ids[-max_prompt_len:]

        input_ids = prompt_ids + completion_ids
        labels = [-100] * len(prompt_ids) + completion_ids.copy()
        attention_mask = [1] * len(input_ids)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "source_id": safe_cell_value(row, "source_id", ""),
            "draft_group": safe_cell_value(row, "draft_group", ""),
        }


def corrector_data_collator(features):
    max_len = max(len(f["input_ids"]) for f in features)
    pad_id = tokenizer.pad_token_id

    batch_input_ids = []
    batch_attention_mask = []
    batch_labels = []

    for f in features:
        pad_len = max_len - len(f["input_ids"])

        batch_input_ids.append(f["input_ids"] + [pad_id] * pad_len)
        batch_attention_mask.append(f["attention_mask"] + [0] * pad_len)
        batch_labels.append(f["labels"] + [-100] * pad_len)

    return {
        "input_ids": torch.tensor(batch_input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(batch_attention_mask, dtype=torch.long),
        "labels": torch.tensor(batch_labels, dtype=torch.long),
    }


# Reload from disk if runtime restarted after Cell 9B.
if "corrector_train_df" not in globals():
    corrector_train_df = pd.read_csv(CORRECTOR_TRAIN_PATH)

if "corrector_eval_df" not in globals():
    corrector_eval_df = pd.read_csv(EVAL_DRAFT_PATH)

required_train_cols = {
    "source_id",
    "source_text",
    "target_arabic",
    "lora_draft",
    "few_shot_block",
    "metadata_block",
    "previous_context_block",
    "draft_group",
    "draft_chrF++",
    "draft_spBLEU",
    "arabic_ratio",
    "han_ratio",
    "latin_ratio",
    "has_chinese",
    "mostly_non_arabic",
    "empty_draft",
    "garbage_draft",
}

missing_train_cols = required_train_cols - set(corrector_train_df.columns)

if missing_train_cols:
    raise ValueError(
        f"corrector_train_df is missing required columns: {missing_train_cols}. "
        "Rerun Cell 9B."
    )

corrector_train_dataset = CorrectorSFTDataset(
    corrector_train_df,
    tokenizer=tokenizer,
    max_length=MAX_SEQ_LENGTH,
)

# This eval dataset is only for optional NLL sanity checks, not for training.
corrector_eval_nll_df = corrector_eval_df.head(EVAL_NLL_LIMIT).copy()
corrector_eval_dataset = CorrectorSFTDataset(
    corrector_eval_nll_df,
    tokenizer=tokenizer,
    max_length=MAX_SEQ_LENGTH,
)

print("Selected corrector train rows:", len(corrector_train_df))
print("corrector_train_dataset:", len(corrector_train_dataset))
print("corrector_eval_dataset for optional NLL sanity:", len(corrector_eval_dataset))
print("Selection metric:", CORRECTOR_SELECTION_METRIC)
print("Selection fraction:", CORRECTOR_BAD_FRACTION)
print("Router train group counts:")
display(corrector_train_df["draft_group"].value_counts(dropna=False).to_frame("count"))

example_item = corrector_train_dataset[0]
print("\nExample token lengths:")
print("input_ids:", len(example_item["input_ids"]))
print("trained target tokens:", int(sum(x != -100 for x in example_item["labels"])))

print("\nExample corrector prompt:")
print(make_corrector_prompt(corrector_train_df.iloc[0].to_dict())[:3000])

print("\nExample corrector completion:")
print(make_corrector_completion(corrector_train_df.iloc[0].to_dict())[:500])


Selected corrector train rows: 3108
corrector_train_dataset: 3108
corrector_eval_dataset for optional NLL sanity: 64
Selection metric: chrF++
Selection fraction: 0.3
Router train group counts:


,count
draft_group,
bad_arabic,1554
retain,777
garbage,777



Example token lengths:
input_ids: 844
trained target tokens: 50

Example corrector prompt:
### System:
You are a professional Egyptian Arabic machine translation router-corrector. You receive the original English dialogue turn, context, metadata, examples, and a draft Arabic translation from a first-stage LoRA model. Your job is to decide whether to correct the draft, ignore it, or keep it close. Return only the final Egyptian Arabic translation, without explanation.

### Instruction:
Task:
Produce the final Egyptian Arabic translation for the current English dialogue turn.
You are given the original source/context and a first-stage LoRA draft.
This is a router-corrector task, not a blind draft editor.

Few-shot training examples:
Example 1 (config=EG, dialect=Egyptian Arabic (Cairene) Dialect, domain=Professional and workplace)
English:
Your Excellency, we have thoroughly reviewed the tender specifications. Our proposal not only meets all technical requirements but also guarantees pr

### Initialize LoRA router-corrector adapter


In [14]:
# ============================================================
# Cell 11 — Initialize second-stage LoRA router-corrector adapter
# Simpler PEFT-safe version:
# Train only router_corrector adapter using cached LoRA-700 drafts in the prompt
# ============================================================

import torch
from peft import LoraConfig, TaskType, prepare_model_for_kbit_training

assert STAGE2_MODE == "lora_router_corrector", (
    f"This cell is only for STAGE2_MODE='lora_router_corrector', got {STAGE2_MODE}"
)

if not hasattr(model, "peft_config"):
    raise TypeError(
        "Expected model to be a PEFT model loaded in Cell 8. "
        "Run Cell 8 first."
    )

print("Available adapters before adding corrector:", list(model.peft_config.keys()))

# ------------------------------------------------------------
# Important design decision
# ------------------------------------------------------------
# This PEFT runtime does not support adapter stacking:
#     model.set_adapter(["default", "router_corrector"])
#
# So we do NOT try to activate first-stage LoRA weights together with
# the second-stage adapter.
#
# This is acceptable for the router-corrector experiment because the
# first-stage LoRA-700 draft is already included in every training/eval prompt:
#
#     source/context + LoRA-700 draft -> gold target_arabic
#
# So the stage-1 signal is the cached draft, not the active adapter weights.
# ------------------------------------------------------------

STAGE2_USES_CACHED_DRAFT_ONLY = True

print("STAGE2_USES_CACHED_DRAFT_ONLY:", STAGE2_USES_CACHED_DRAFT_ONLY)
print("First-stage LoRA signal is provided through lora_draft column.")

# ------------------------------------------------------------
# Freeze everything first
# ------------------------------------------------------------

for name, param in model.named_parameters():
    param.requires_grad = False

# ------------------------------------------------------------
# Prepare quantized model for LoRA training
# ------------------------------------------------------------

if LOAD_IN_4BIT:
    model = prepare_model_for_kbit_training(
        model,
        use_gradient_checkpointing=True,
    )
    print("prepare_model_for_kbit_training applied.")

# ------------------------------------------------------------
# Recreate router_corrector adapter cleanly if it exists from failed Cell 11
# ------------------------------------------------------------

if CORRECTOR_ADAPTER_NAME in model.peft_config:
    print(f"Deleting existing in-memory adapter '{CORRECTOR_ADAPTER_NAME}' from failed/previous init.")
    try:
        model.delete_adapter(CORRECTOR_ADAPTER_NAME)
    except Exception as e:
        raise RuntimeError(
            f"Could not delete existing adapter '{CORRECTOR_ADAPTER_NAME}'. "
            "Restart runtime and rerun from Cell 1 if this persists."
        ) from e

print("Adapters after cleanup:", list(model.peft_config.keys()))

corrector_lora_config = LoraConfig(
    r=CORRECTOR_LORA_R,
    lora_alpha=CORRECTOR_LORA_ALPHA,
    lora_dropout=CORRECTOR_LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=CORRECTOR_LORA_TARGET_MODULES,
)

model.add_adapter(CORRECTOR_ADAPTER_NAME, corrector_lora_config)

print(f"Added fresh second-stage LoRA adapter: {CORRECTOR_ADAPTER_NAME}")
print("Available adapters now:", list(model.peft_config.keys()))

# ------------------------------------------------------------
# Activate only router_corrector adapter
# ------------------------------------------------------------

def set_stage2_active_adapters(train_corrector=True, corrector_adapter_name=None):
    """
    PEFT-safe single-adapter mode.

    We activate only the second-stage router_corrector adapter.
    The first-stage LoRA-700 information is already present as lora_draft
    in the prompt, so we do not require adapter stacking.
    """
    if corrector_adapter_name is None:
        corrector_adapter_name = CORRECTOR_ADAPTER_NAME

    if corrector_adapter_name not in model.peft_config:
        raise RuntimeError(
            f"Corrector adapter '{corrector_adapter_name}' not found. "
            f"Available adapters: {list(model.peft_config.keys())}"
        )

    model.set_adapter(corrector_adapter_name)

    for name, param in model.named_parameters():
        if corrector_adapter_name in name:
            param.requires_grad = bool(train_corrector)
        else:
            param.requires_grad = False

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())

    print("Active adapter:", corrector_adapter_name)
    print("Mode: cached-draft router-corrector, no adapter stacking")
    print(f"Trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.4f}%)")

    return corrector_adapter_name


ACTIVE_ADAPTERS_MODE = set_stage2_active_adapters(train_corrector=True)

model.config.use_cache = False

if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()
    print("Gradient checkpointing enabled.")

if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()
    print("Input gradients enabled.")

print("Stage-2 LoRA router-corrector adapter is ready.")

Available adapters before adding corrector: ['default']
STAGE2_USES_CACHED_DRAFT_ONLY: True
First-stage LoRA signal is provided through lora_draft column.
prepare_model_for_kbit_training applied.
Adapters after cleanup: ['default']
Added fresh second-stage LoRA adapter: router_corrector
Available adapters now: ['default', 'router_corrector']
Active adapter: router_corrector
Mode: cached-draft router-corrector, no adapter stacking
Trainable params: 16,515,072 / 2,255,355,392 (0.7323%)
Gradient checkpointing enabled.
Input gradients enabled.
Stage-2 LoRA router-corrector adapter is ready.


### Check existing LoRA router-corrector checkpoints


In [15]:
# ============================================================
# Cell 12 — Check for existing LoRA router-corrector checkpoints
# Same resumable style as previous second-stage notebooks
# ============================================================

from pathlib import Path
import re


def checkpoint_step(path):
    m = re.search(r"checkpoint-(\d+)", str(path))
    return int(m.group(1)) if m else -1


def corrector_adapter_dir_from_checkpoint(ckpt_dir):
    ckpt_dir = Path(ckpt_dir)

    preferred = ckpt_dir / CORRECTOR_ADAPTER_SAVE_SUBDIR

    if (preferred / "adapter_config.json").exists():
        return preferred

    # Fallback: Trainer may also save adapter files at the checkpoint root.
    if (ckpt_dir / "adapter_config.json").exists():
        return ckpt_dir

    # Fallback for PEFT multi-adapter save format.
    named = ckpt_dir / CORRECTOR_ADAPTER_NAME
    if (named / "adapter_config.json").exists():
        return named

    return preferred


def list_corrector_checkpoints(output_dir):
    output_dir = Path(output_dir)

    checkpoints = []

    for p in output_dir.glob("checkpoint-*"):
        if not p.is_dir():
            continue

        step = checkpoint_step(p)
        adapter_dir = corrector_adapter_dir_from_checkpoint(p)

        has_adapter_config = (adapter_dir / "adapter_config.json").exists()
        has_adapter_weights = (
            (adapter_dir / "adapter_model.safetensors").exists()
            or (adapter_dir / "adapter_model.bin").exists()
        )

        if step >= 0 and has_adapter_config and has_adapter_weights:
            checkpoints.append((step, p, adapter_dir))

    checkpoints = sorted(checkpoints, key=lambda x: x[0])
    return checkpoints


def get_last_corrector_checkpoint(output_dir):
    checkpoints = list_corrector_checkpoints(output_dir)
    return checkpoints[-1] if checkpoints else None


existing_corrector_checkpoints = list_corrector_checkpoints(OUTPUT_DIR)

print("Existing LoRA router-corrector checkpoints:")
if existing_corrector_checkpoints:
    for step, ckpt_dir, adapter_dir in existing_corrector_checkpoints:
        print(f"  step={step:>6} | ckpt={ckpt_dir} | adapter={adapter_dir}")
else:
    print("  none")

last_corrector_checkpoint = get_last_corrector_checkpoint(OUTPUT_DIR)
print("\nLast corrector checkpoint:", last_corrector_checkpoint)


Existing LoRA router-corrector checkpoints:
  step=    50 | ckpt=/content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_lora_step700_alexandria_eg_only_context3_complete2shot_LORA_ROUTER_CORRECTOR_draft2gold_fulltrain_mix_badarabic_garbage_retain_worst0p3_chrfpp_fulleval_r8_alpha16_drop0p05_lr5e-05_3epochs/checkpoint-50 | adapter=/content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_lora_step700_alexandria_eg_only_context3_complete2shot_LORA_ROUTER_CORRECTOR_draft2gold_fulltrain_mix_badarabic_garbage_retain_worst0p3_chrfpp_fulleval_r8_alpha16_drop0p05_lr5e-05_3epochs/checkpoint-50
  step=   100 | ckpt=/content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_lora_step700_alexandria_eg_only_context3_complete2shot_LORA_ROUTER_CORRECTOR_draft2gold_fulltrain_mix_badarabic_garbage_retain_worst0p3_chrfpp_fulleval_r8_alpha16_drop0p05_lr5e-05_3epochs/checkpoint-100 | adapter=/content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_lora_step700_alexandria_eg_only_context3_complete2s

### **Trainer, scheduler, and checkpoint helpers**


In [16]:
# ============================================================
# Cell 13 — Trainer/checkpoint helpers for LoRA router-corrector
# Version-safe TrainingArguments for current Colab/Transformers
# ============================================================

import json
import inspect
from pathlib import Path

import torch
from transformers import Trainer, TrainingArguments, TrainerCallback


# ------------------------------------------------------------
# Save adapter-only helper
# ------------------------------------------------------------

def save_corrector_adapter_only(save_dir, model_to_save=None):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    model_to_save = model if model_to_save is None else model_to_save

    # PEFT versions differ in save_pretrained(selected_adapters=...)
    # Try selected_adapters first; fallback to normal save_pretrained.
    try:
        model_to_save.save_pretrained(
            str(save_dir),
            selected_adapters=[CORRECTOR_ADAPTER_NAME],
            safe_serialization=True,
        )
    except TypeError:
        model_to_save.save_pretrained(
            str(save_dir),
            safe_serialization=True,
        )

    metadata = {
        "stage2_mode": STAGE2_MODE,
        "experiment_name": EXPERIMENT_NAME,
        "base_model": MODEL_NAME,
        "first_stage_lora_adapter_path": str(LORA_ADAPTER_PATH),
        "first_stage_lora_step": int(MANUAL_LORA_BEST_STEP),
        "first_stage_adapter_name": FIRST_STAGE_ADAPTER_NAME,
        "corrector_adapter_name": CORRECTOR_ADAPTER_NAME,
        "stage2_uses_cached_draft_only": True,
        "corrector_lora_r": CORRECTOR_LORA_R,
        "corrector_lora_alpha": CORRECTOR_LORA_ALPHA,
        "corrector_lora_dropout": CORRECTOR_LORA_DROPOUT,
        "corrector_lr": CORRECTOR_LR,
        "corrector_num_epochs": CORRECTOR_NUM_EPOCHS,
        "max_seq_length": MAX_SEQ_LENGTH,
        "max_new_tokens": MAX_NEW_TOKENS,
        "corrector_train_path": str(CORRECTOR_TRAIN_PATH),
    }

    with open(save_dir / "router_corrector_metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)


class SaveCorrectorAdapterCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        checkpoint_dir = Path(args.output_dir) / f"checkpoint-{state.global_step}"
        adapter_dir = checkpoint_dir / CORRECTOR_ADAPTER_SAVE_SUBDIR

        save_corrector_adapter_only(
            adapter_dir,
            model_to_save=kwargs["model"],
        )

        print("Saved adapter-only checkpoint:", adapter_dir)

        return control


# ------------------------------------------------------------
# Version-safe TrainingArguments builder
# ------------------------------------------------------------

def build_training_arguments_version_safe():
    sig = inspect.signature(TrainingArguments.__init__)
    supported = set(sig.parameters.keys())

    requested = {
        "output_dir": str(OUTPUT_DIR),

        # Do NOT include overwrite_output_dir.
        # Your current TrainingArguments does not support it.

        "num_train_epochs": CORRECTOR_NUM_EPOCHS,
        "per_device_train_batch_size": PER_DEVICE_BATCH_SIZE,
        "gradient_accumulation_steps": GRAD_ACCUM_STEPS,

        "learning_rate": CORRECTOR_LR,
        "warmup_ratio": WARMUP_RATIO,
        "weight_decay": WEIGHT_DECAY,
        "lr_scheduler_type": "cosine",

        "logging_steps": LOGGING_STEPS,
        "save_steps": SAVE_STEPS,
        "save_total_limit": SAVE_TOTAL_LIMIT,

        "fp16": torch.cuda.is_available(),
        "bf16": False,

        "optim": "paged_adamw_8bit" if LOAD_IN_4BIT else "adamw_torch",

        "report_to": "none",
        "remove_unused_columns": False,
        "gradient_checkpointing": True,
    }

    # Some versions use evaluation_strategy, newer ones may use eval_strategy.
    if "evaluation_strategy" in supported:
        requested["evaluation_strategy"] = "no"
    elif "eval_strategy" in supported:
        requested["eval_strategy"] = "no"

    # Some versions support save_strategy.
    if "save_strategy" in supported:
        requested["save_strategy"] = "steps"

    # Keep only args supported by the installed TrainingArguments.
    filtered = {
        k: v
        for k, v in requested.items()
        if k in supported
    }

    dropped = sorted(set(requested.keys()) - set(filtered.keys()))

    print("TrainingArguments supported args used:")
    print(sorted(filtered.keys()))

    if dropped:
        print("\nDropped unsupported TrainingArguments args:")
        print(dropped)

    return TrainingArguments(**filtered)


training_args = build_training_arguments_version_safe()

print("\nTrainingArguments created successfully:")
print(training_args)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


TrainingArguments supported args used:
['bf16', 'eval_strategy', 'fp16', 'gradient_accumulation_steps', 'gradient_checkpointing', 'learning_rate', 'logging_steps', 'lr_scheduler_type', 'num_train_epochs', 'optim', 'output_dir', 'per_device_train_batch_size', 'remove_unused_columns', 'report_to', 'save_steps', 'save_strategy', 'save_total_limit', 'warmup_ratio', 'weight_decay']

TrainingArguments created successfully:
TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,

### **Train / resume LoRA router-corrector**


In [17]:
# ============================================================
# Cell 14 — Initialize or resume LoRA router-corrector Trainer
# Same resume-safe flow as previous second-stage notebooks
# ============================================================



CORRECTOR_TRAINING_ARGS = training_args

# Dataset/collator compatibility, depending on which Cell 10 version you used.
if "corrector_train_dataset" not in globals() and "train_dataset" in globals():
    corrector_train_dataset = train_dataset

if "corrector_data_collator" not in globals() and "data_collator" in globals():
    corrector_data_collator = data_collator

print("✅ Compatibility aliases ready.")
print("CORRECTOR_TRAINING_ARGS:", type(CORRECTOR_TRAINING_ARGS))
print("corrector_train_dataset rows:", len(corrector_train_dataset))
print("corrector_data_collator:", type(corrector_data_collator))
import math

# Make sure the adapter state is correct before Trainer sees the model.
ACTIVE_ADAPTERS_MODE = set_stage2_active_adapters(train_corrector=True)
model.train()

last_corrector_checkpoint = get_last_corrector_checkpoint(OUTPUT_DIR)

if last_corrector_checkpoint is not None:
    LAST_CORRECTOR_STEP, LAST_CORRECTOR_CKPT_DIR, LAST_CORRECTOR_ADAPTER_DIR = last_corrector_checkpoint
    CORRECTOR_RESUME_CHECKPOINT = str(LAST_CORRECTOR_CKPT_DIR)
    print("Resuming LoRA router-corrector from:", CORRECTOR_RESUME_CHECKPOINT)
    print("Adapter snapshot:", LAST_CORRECTOR_ADAPTER_DIR)
else:
    LAST_CORRECTOR_STEP = 0
    LAST_CORRECTOR_CKPT_DIR = None
    LAST_CORRECTOR_ADAPTER_DIR = None
    CORRECTOR_RESUME_CHECKPOINT = None
    print("No existing LoRA router-corrector checkpoint. Starting from scratch.")

steps_per_epoch = math.ceil(len(corrector_train_dataset) / max(1, PER_DEVICE_BATCH_SIZE * GRAD_ACCUM_STEPS))
expected_total_steps = int(steps_per_epoch * CORRECTOR_NUM_EPOCHS)

trainer = Trainer(
    model=model,
    args=CORRECTOR_TRAINING_ARGS,
    train_dataset=corrector_train_dataset,
    data_collator=corrector_data_collator,
    callbacks=[SaveCorrectorAdapterCallback()],
)

print("\nTrainer ready.")
print("Active adapters mode:", ACTIVE_ADAPTERS_MODE)
print("Start step:", LAST_CORRECTOR_STEP)
print("Expected total optimizer steps:", expected_total_steps)
print("Resume checkpoint:", CORRECTOR_RESUME_CHECKPOINT)


✅ Compatibility aliases ready.
CORRECTOR_TRAINING_ARGS: <class 'transformers.training_args.TrainingArguments'>
corrector_train_dataset rows: 3108
corrector_data_collator: <class 'function'>
Active adapter: router_corrector
Mode: cached-draft router-corrector, no adapter stacking
Trainable params: 16,515,072 / 2,255,355,392 (0.7323%)
Resuming LoRA router-corrector from: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_lora_step700_alexandria_eg_only_context3_complete2shot_LORA_ROUTER_CORRECTOR_draft2gold_fulltrain_mix_badarabic_garbage_retain_worst0p3_chrfpp_fulleval_r8_alpha16_drop0p05_lr5e-05_3epochs/checkpoint-950
Adapter snapshot: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_lora_step700_alexandria_eg_only_context3_complete2shot_LORA_ROUTER_CORRECTOR_draft2gold_fulltrain_mix_badarabic_garbage_retain_worst0p3_chrfpp_fulleval_r8_alpha16_drop0p05_lr5e-05_3epochs/checkpoint-950

Trainer ready.
Active adapters mode: router_corrector
Start step: 950
Expected total 

### **Sweep LoRA router-corrector checkpoints by generation metrics**


In [18]:
# ============================================================
# Cell 15 — Train or resume LoRA router-corrector
# Saves adapter checkpoints every SAVE_STEPS under a separate LoRA-corrector path
# ============================================================

import json
from pathlib import Path

ACTIVE_ADAPTERS_MODE = set_stage2_active_adapters(train_corrector=True)
model.train()

train_result = trainer.train(resume_from_checkpoint=CORRECTOR_RESUME_CHECKPOINT)

print("\nTraining finished.")
print(train_result)

# Save final adapter-only copy in OUTPUT_DIR. Metric-best selection still happens
# by generated metrics in Cell 16.
FINAL_RUN_ADAPTER_DIR = OUTPUT_DIR / "final_router_corrector_adapter_last"
save_corrector_adapter_only(FINAL_RUN_ADAPTER_DIR, model_to_save=model)

trainer_state_path = OUTPUT_DIR / "router_corrector_trainer_result.json"
trainer_state_payload = {
    "train_result": str(train_result),
    "stage2_mode": STAGE2_MODE,
    "experiment_name": EXPERIMENT_NAME,
    "final_run_adapter_dir": str(FINAL_RUN_ADAPTER_DIR),
    "corrector_train_path": str(CORRECTOR_TRAIN_PATH),
    "diagnostic_counts_path": str(DIAGNOSTIC_COUNTS_PATH),
}

trainer_state_path.write_text(
    json.dumps(trainer_state_payload, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

# Refresh checkpoint list after training.
existing_corrector_checkpoints = list_corrector_checkpoints(OUTPUT_DIR)

print("\nSaved final last-step corrector adapter:", FINAL_RUN_ADAPTER_DIR)
print("Saved training result:", trainer_state_path)
print("\nAvailable checkpoints after training:")
for step, ckpt_dir, adapter_dir in existing_corrector_checkpoints:
    print(f"  step={step:>6} | ckpt={ckpt_dir} | adapter={adapter_dir}")


Active adapter: router_corrector
Mode: cached-draft router-corrector, no adapter stacking
Trainable params: 16,515,072 / 2,255,355,392 (0.7323%)


Step,Training Loss
960,0.588587
970,0.649560
980,0.663747
990,0.720881
1000,0.554884
1010,0.599586
1020,0.700357
1030,0.621928
1040,0.494628
1050,0.629241


config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

Saved adapter-only checkpoint: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_lora_step700_alexandria_eg_only_context3_complete2shot_LORA_ROUTER_CORRECTOR_draft2gold_fulltrain_mix_badarabic_garbage_retain_worst0p3_chrfpp_fulleval_r8_alpha16_drop0p05_lr5e-05_3epochs/checkpoint-1000/router_corrector_adapter
Saved adapter-only checkpoint: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_lora_step700_alexandria_eg_only_context3_complete2shot_LORA_ROUTER_CORRECTOR_draft2gold_fulltrain_mix_badarabic_garbage_retain_worst0p3_chrfpp_fulleval_r8_alpha16_drop0p05_lr5e-05_3epochs/checkpoint-1050/router_corrector_adapter


KeyboardInterrupt: 

### **Save metric-best LoRA router-corrector**


In [19]:
# ============================================================
# Cell 16A0 — Baseline metrics for LoRA-700 drafts on full eval
# Full eval-aware version + garbage/non-garbage subset metrics
# ============================================================

import json
import pandas as pd
from sacrebleu.metrics import BLEU, CHRF


def compute_lexical_metrics_for_lists(preds, refs):
    bleu_metric = BLEU(tokenize="13a")
    spbleu_metric = BLEU(tokenize="flores200")
    chrf_metric = CHRF(word_order=0)
    chrfpp_metric = CHRF(word_order=2)

    return {
        "BLEU": float(bleu_metric.corpus_score(preds, [refs]).score),
        "spBLEU": float(spbleu_metric.corpus_score(preds, [refs]).score),
        "chrF": float(chrf_metric.corpus_score(preds, [refs]).score),
        "chrF++": float(chrfpp_metric.corpus_score(preds, [refs]).score),
    }


def prefixed_metrics(prefix, preds, refs):
    metrics = compute_lexical_metrics_for_lists(preds, refs)
    return {f"{prefix}_{k}": v for k, v in metrics.items()}


if "corrector_eval_df" not in globals():
    corrector_eval_df = pd.read_csv(EVAL_DRAFT_PATH)

corrector_eval_df["source_id"] = corrector_eval_df["source_id"].astype(str)
corrector_eval_df["lora_draft"] = corrector_eval_df["lora_draft"].fillna("").astype(str)
corrector_eval_df["target_arabic"] = corrector_eval_df["target_arabic"].fillna("").astype(str)

# Backfill diagnostics if this file was loaded from an older cache.
if "garbage_draft" not in corrector_eval_df.columns:
    corrector_eval_df = add_draft_diagnostics(corrector_eval_df)
    corrector_eval_df["draft_group"] = np.where(
        corrector_eval_df["garbage_draft"].astype(bool),
        "garbage",
        "non_garbage",
    )
    corrector_eval_df.to_csv(EVAL_DRAFT_PATH, index=False, encoding="utf-8-sig")

draft_preds = corrector_eval_df["lora_draft"].tolist()
draft_refs = corrector_eval_df["target_arabic"].tolist()

lora_draft_baseline_metrics = {
    "system": f"Qwen3-4B-LoRA-step{LORA_BEST_STEP}-draft-only",
    "experiment_name": EXPERIMENT_NAME,
    "n_examples": len(draft_preds),
    "is_full_eval": bool(len(corrector_eval_df) == len(eval_df)),
    "corrector_selection_metric": CORRECTOR_SELECTION_METRIC,
    "corrector_bad_fraction": CORRECTOR_BAD_FRACTION,
    "corrector_train_examples": int(len(corrector_train_df)) if "corrector_train_df" in globals() else None,
    "eval_garbage_drafts": int(corrector_eval_df["garbage_draft"].astype(bool).sum()),
    "eval_non_garbage_drafts": int((~corrector_eval_df["garbage_draft"].astype(bool)).sum()),
    **compute_lexical_metrics_for_lists(draft_preds, draft_refs),
}

for subset_name, mask in {
    "garbage": corrector_eval_df["garbage_draft"].astype(bool),
    "non_garbage": ~corrector_eval_df["garbage_draft"].astype(bool),
}.items():
    subset_df = corrector_eval_df[mask].copy()

    if len(subset_df) == 0:
        lora_draft_baseline_metrics[f"{subset_name}_n"] = 0
        continue

    lora_draft_baseline_metrics[f"{subset_name}_n"] = int(len(subset_df))
    lora_draft_baseline_metrics.update(
        prefixed_metrics(
            f"{subset_name}_draft",
            subset_df["lora_draft"].astype(str).tolist(),
            subset_df["target_arabic"].astype(str).tolist(),
        )
    )

LORA_DRAFT_BASELINE_METRICS_PATH = (
    CORRECTOR_CACHE_DIR
    / f"lora_step{LORA_BEST_STEP}_draft_baseline_metrics_fulleval.json"
)

LORA_DRAFT_BASELINE_METRICS_PATH.write_text(
    json.dumps(lora_draft_baseline_metrics, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(json.dumps(lora_draft_baseline_metrics, indent=2, ensure_ascii=False))
print("\nSaved:", LORA_DRAFT_BASELINE_METRICS_PATH)

if len(corrector_eval_df) != len(eval_df):
    print(
        "\nWARNING: This is not full eval. "
        f"corrector_eval_df={len(corrector_eval_df)}, eval_df={len(eval_df)}"
    )
else:
    print("\nFull eval baseline confirmed.")


{
  "system": "Qwen3-4B-LoRA-step700-draft-only",
  "experiment_name": "qwen3_4b_lora_step700_alexandria_eg_only_context3_complete2shot_LORA_ROUTER_CORRECTOR_draft2gold_fulltrain_mix_badarabic_garbage_retain_worst0p3_chrfpp_fulleval_r8_alpha16_drop0p05_lr5e-05_3epochs",
  "n_examples": 1118,
  "is_full_eval": true,
  "corrector_selection_metric": "chrF++",
  "corrector_bad_fraction": 0.3,
  "corrector_train_examples": 3108,
  "eval_garbage_drafts": 3,
  "eval_non_garbage_drafts": 1115,
  "BLEU": 10.176410257229774,
  "spBLEU": 19.371658933026904,
  "chrF": 38.105813746733794,
  "chrF++": 35.35988931136216,
  "garbage_n": 3,
  "garbage_draft_BLEU": 9.811596837407793,
  "garbage_draft_spBLEU": 15.907087384598421,
  "garbage_draft_chrF": 30.152480590341995,
  "garbage_draft_chrF++": 29.17249643102825,
  "non_garbage_n": 1115,
  "non_garbage_draft_BLEU": 10.176393042257745,
  "non_garbage_draft_spBLEU": 19.379977133183,
  "non_garbage_draft_chrF": 38.12480354540931,
  "non_garbage_draft_ch

### **Second Stage: LoRA router-corrector**


In [ ]:
# ============================================================
# Cell 16 — Sweep second-stage LoRA router-corrector checkpoints by generated metrics
# Strict sweep: only checkpoints 200, 500, 800, 1000
# Resumable by source_id after session restart
# Select by chrF++ first, spBLEU second
# ============================================================

import re
import json
import pandas as pd
import torch
from pathlib import Path
from sacrebleu.metrics import BLEU, CHRF
from tqdm.auto import tqdm

# ------------------------------------------------------------
# Strict checkpoint list
# ------------------------------------------------------------

TARGET_SWEEP_STEPS = [200, 500, 800, 1000]

print("Strict target sweep steps:", TARGET_SWEEP_STEPS)


# ------------------------------------------------------------
# Text cleanup + metrics
# ------------------------------------------------------------

def clean_generated_answer(text):
    text = "" if text is None else str(text)

    if RESPONSE_MARKER in text:
        text = text.split(RESPONSE_MARKER)[-1]

    special_tokens = [
        tokenizer.eos_token,
        tokenizer.pad_token,
        "<|endoftext|>",
        "<|im_end|>",
        "<|end|>",
    ]

    for tok in special_tokens:
        if tok:
            text = text.replace(tok, "")

    stop_markers = [
        SYSTEM_MARKER,
        INSTRUCTION_MARKER,
        RESPONSE_MARKER,
        "### System",
        "### Instruction",
        "### Arabic translation",
        "Final Egyptian Arabic translation:",
    ]

    for marker in stop_markers:
        if marker in text:
            text = text.split(marker)[0]

    text = re.sub(r"\s+", " ", text).strip()

    return text


def compute_lexical_metrics(preds, refs):
    preds = ["" if pd.isna(x) else str(x) for x in preds]
    refs = ["" if pd.isna(x) else str(x) for x in refs]

    bleu_metric = BLEU(tokenize="13a")
    spbleu_metric = BLEU(tokenize="flores200")
    chrf_metric = CHRF(word_order=0)
    chrfpp_metric = CHRF(word_order=2)

    return {
        "BLEU": float(bleu_metric.corpus_score(preds, [refs]).score),
        "spBLEU": float(spbleu_metric.corpus_score(preds, [refs]).score),
        "chrF": float(chrf_metric.corpus_score(preds, [refs]).score),
        "chrF++": float(chrfpp_metric.corpus_score(preds, [refs]).score),
    }


# ------------------------------------------------------------
# Robust checkpoint discovery
# ------------------------------------------------------------

def get_checkpoint_step(path):
    m = re.search(r"checkpoint-(\d+)", str(path))
    return int(m.group(1)) if m else -1


def infer_corrector_adapter_dir(ckpt_dir):
    ckpt_dir = Path(ckpt_dir)

    candidates = [
        ckpt_dir / CORRECTOR_ADAPTER_SAVE_SUBDIR,
        ckpt_dir / CORRECTOR_ADAPTER_NAME,
        ckpt_dir,
    ]

    for p in candidates:
        if (p / "adapter_config.json").exists():
            return p

    # Return preferred path even if missing, so the error message is clear.
    return ckpt_dir / CORRECTOR_ADAPTER_SAVE_SUBDIR


def normalize_available_checkpoints():
    """
    Works with either:
    - existing list_corrector_checkpoints(OUTPUT_DIR) returning tuples:
      (step, ckpt_dir, adapter_dir)
    - or dict rows:
      {"step": ..., "checkpoint_dir": ..., "adapter_dir": ...}
    - or no helper at all, in which case it scans OUTPUT_DIR/checkpoint-*.
    """
    rows = []

    if "list_corrector_checkpoints" in globals():
        raw = list_corrector_checkpoints(OUTPUT_DIR)

        for item in raw:
            if isinstance(item, dict):
                step = int(item.get("step", -1))
                ckpt_dir = Path(item.get("checkpoint_dir"))
                adapter_dir = Path(
                    item.get("adapter_dir")
                    or item.get("corrector_adapter_dir")
                    or infer_corrector_adapter_dir(ckpt_dir)
                )
            else:
                step = int(item[0])
                ckpt_dir = Path(item[1])
                adapter_dir = Path(item[2]) if len(item) >= 3 else infer_corrector_adapter_dir(ckpt_dir)

            if step >= 0:
                rows.append((step, ckpt_dir, adapter_dir))

    else:
        for ckpt_dir in sorted(Path(OUTPUT_DIR).glob("checkpoint-*"), key=get_checkpoint_step):
            step = get_checkpoint_step(ckpt_dir)
            if step >= 0:
                adapter_dir = infer_corrector_adapter_dir(ckpt_dir)
                rows.append((step, ckpt_dir, adapter_dir))

    # Deduplicate by step and keep the last discovered version.
    by_step = {}

    for step, ckpt_dir, adapter_dir in rows:
        by_step[int(step)] = (Path(ckpt_dir), Path(adapter_dir))

    return by_step


available_by_step = normalize_available_checkpoints()

print("Available corrector checkpoint steps:", sorted(available_by_step.keys()))

missing_target_steps = [
    s for s in TARGET_SWEEP_STEPS
    if s not in available_by_step
]

steps_to_sweep = [
    s for s in TARGET_SWEEP_STEPS
    if s in available_by_step
]

if missing_target_steps:
    print("Missing requested checkpoints, will skip:", missing_target_steps)

if not steps_to_sweep:
    raise FileNotFoundError(
        f"None of the requested checkpoints {TARGET_SWEEP_STEPS} were found under {OUTPUT_DIR}. "
        "Run training longer or adjust TARGET_SWEEP_STEPS."
    )

print("Final steps to sweep:", steps_to_sweep)


# ------------------------------------------------------------
# Adapter loading
# ------------------------------------------------------------

def load_corrector_adapter_for_inference(adapter_dir, adapter_name=None):
    adapter_dir = Path(adapter_dir)
    adapter_name = adapter_name or CORRECTOR_ADAPTER_NAME

    if not (adapter_dir / "adapter_config.json").exists():
        raise FileNotFoundError(f"Missing adapter_config.json in {adapter_dir}")

    # Try to reload the same adapter name cleanly.
    if adapter_name in model.peft_config:
        try:
            model.delete_adapter(adapter_name)
            print(f"Deleted existing adapter before reload: {adapter_name}")
        except Exception as e:
            unique_adapter_name = f"{adapter_name}_loaded_{abs(hash(str(adapter_dir))) % 10**8}"
            print(
                f"Could not delete existing adapter {adapter_name!r}: {repr(e)}. "
                f"Loading checkpoint as {unique_adapter_name!r}."
            )
            adapter_name = unique_adapter_name

    if adapter_name not in model.peft_config:
        model.load_adapter(
            str(adapter_dir),
            adapter_name=adapter_name,
            is_trainable=False,
        )

    set_stage2_active_adapters(
        train_corrector=False,
        corrector_adapter_name=adapter_name,
    )

    model.eval()

    return adapter_name


# ------------------------------------------------------------
# Generation
# ------------------------------------------------------------

def generate_with_lora_corrector_from_row(row, max_new_tokens=MAX_NEW_TOKENS):
    prompt = make_corrector_prompt(row)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(device)

    input_len = inputs["input_ids"].shape[-1]

    model.eval()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated_ids = outputs[0, input_len:]
    decoded_new = tokenizer.decode(generated_ids, skip_special_tokens=False)

    return clean_generated_answer(decoded_new)


# ------------------------------------------------------------
# Build full sweep eval set
# ------------------------------------------------------------

if "corrector_eval_df" not in globals():
    corrector_eval_df = pd.read_csv(EVAL_DRAFT_PATH)

sweep_eval_df = corrector_eval_df.reset_index(drop=True).copy()

if "source_id" not in sweep_eval_df.columns:
    sweep_eval_df["source_id"] = [f"eval_{i}" for i in range(len(sweep_eval_df))]

sweep_eval_df["source_id"] = sweep_eval_df["source_id"].astype(str)

if len(sweep_eval_df) != len(eval_df):
    raise RuntimeError(
        "Sweep must use full eval. "
        f"sweep_eval_df={len(sweep_eval_df)}, eval_df={len(eval_df)}. Rerun Cell 9B."
    )

print("Sweep eval examples:", len(sweep_eval_df))
print("This is the same full eval set used by the LoRA draft baseline.")


# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------

sweep_dir = OUTPUT_DIR / "metric_sweep"
sweep_dir.mkdir(parents=True, exist_ok=True)

sweep_rows = []


# ------------------------------------------------------------
# Sweep only TARGET_SWEEP_STEPS that exist
# ------------------------------------------------------------

for step in steps_to_sweep:
    ckpt_dir, adapter_dir = available_by_step[step]
    pred_path = sweep_dir / f"sweep_predictions_step{step}.csv"

    print("\n==============================")
    print("Sweeping LoRA router-corrector checkpoint step:", step)
    print("Checkpoint directory:", ckpt_dir)
    print("Adapter directory:", adapter_dir)
    print("Prediction file:", pred_path)
    print("==============================")

    active_adapter_name = load_corrector_adapter_for_inference(
        adapter_dir,
        adapter_name=CORRECTOR_ADAPTER_NAME,
    )

    expected_ids = set(sweep_eval_df["source_id"].astype(str).tolist())

    # --------------------------------------------------------
    # Resume existing predictions for this checkpoint
    # --------------------------------------------------------

    if pred_path.exists():
        existing_df = pd.read_csv(pred_path)

        if "source_id" not in existing_df.columns:
            existing_df["source_id"] = [f"eval_{i}" for i in range(len(existing_df))]

        existing_df["source_id"] = existing_df["source_id"].astype(str)

        if existing_df.columns.duplicated().any():
            dup_cols = existing_df.columns[existing_df.columns.duplicated()].tolist()
            print(f"Step {step}: dropping duplicate columns from prediction file:")
            print(dup_cols)
            existing_df = existing_df.loc[:, ~existing_df.columns.duplicated()].copy()

        existing_df = existing_df[
            existing_df["source_id"].isin(expected_ids)
        ].copy()

        existing_df = existing_df.drop_duplicates(
            subset=["source_id"],
            keep="first",
        )

        if "prediction" not in existing_df.columns:
            existing_df["prediction"] = ""

        existing_df["prediction"] = existing_df["prediction"].fillna("").astype(str)

        done_df = existing_df[
            existing_df["prediction"].str.strip().str.len() > 0
        ].copy()

        out_rows = done_df.to_dict("records")
        done_ids = set(done_df["source_id"].astype(str).tolist())

        print(f"Resuming step {step}: {len(done_ids)} / {len(sweep_eval_df)} already done.")

    else:
        out_rows = []
        done_ids = set()
        print(f"No existing predictions for step {step}. Starting fresh.")

    # --------------------------------------------------------
    # Generate only missing rows
    # --------------------------------------------------------

    missing_eval_df = sweep_eval_df[
        ~sweep_eval_df["source_id"].astype(str).isin(done_ids)
    ].reset_index(drop=True)

    print(f"Need to generate for step {step}: {len(missing_eval_df)} remaining examples.")

    generated_since_save = 0

    for _, row in tqdm(
        missing_eval_df.iterrows(),
        total=len(missing_eval_df),
        desc=f"Generate step {step}",
    ):
        row_dict = row.to_dict()
        source_id = str(row_dict.get("source_id", ""))

        try:
            pred = generate_with_lora_corrector_from_row(row_dict)
        except Exception as e:
            pred = ""
            print("Generation failed:", source_id, repr(e))

        ref = str(row_dict.get("target_arabic", row_dict.get("reference_arabic", "")))

        out_rows.append({
            "source_id": source_id,
            "config": row_dict.get("config", ""),
            "domain": row_dict.get("domain", ""),
            "dialect": row_dict.get("dialect", ""),
            "source_text": row_dict.get("source_text", ""),
            "lora_draft": row_dict.get("lora_draft", ""),
            "reference_arabic": ref,
            "prediction": pred,
            "garbage_draft": row_dict.get("garbage_draft", False),
            "draft_group": row_dict.get("draft_group", ""),
        })

        done_ids.add(source_id)
        generated_since_save += 1

        if generated_since_save >= 10:
            tmp_df = pd.DataFrame(out_rows)
            tmp_df["source_id"] = tmp_df["source_id"].astype(str)
            tmp_df = tmp_df.drop_duplicates(subset=["source_id"], keep="first")
            tmp_df.to_csv(pred_path, index=False, encoding="utf-8-sig")
            generated_since_save = 0

    # --------------------------------------------------------
    # Finalize prediction file for this checkpoint
    # --------------------------------------------------------

    out_df = pd.DataFrame(out_rows)

    if len(out_df) == 0:
        raise RuntimeError(f"No predictions generated or loaded for step {step}.")

    if out_df.columns.duplicated().any():
        dup_cols = out_df.columns[out_df.columns.duplicated()].tolist()
        print(f"Step {step}: dropping duplicate columns before finalize:")
        print(dup_cols)
        out_df = out_df.loc[:, ~out_df.columns.duplicated()].copy()

    out_df["source_id"] = out_df["source_id"].astype(str)
    out_df = out_df.drop_duplicates(subset=["source_id"], keep="first")

    order_df = sweep_eval_df[["source_id"]].copy()
    order_df["source_id"] = order_df["source_id"].astype(str)

    out_df = order_df.merge(out_df, on="source_id", how="left")

    meta_cols = [
        "config",
        "domain",
        "dialect",
        "source_text",
        "lora_draft",
        "target_arabic",
        "draft_group",
        "garbage_draft",
        "empty_draft",
        "has_chinese",
        "mostly_non_arabic",
        "arabic_ratio",
        "han_ratio",
        "latin_ratio",
    ]

    meta_cols = [c for c in meta_cols if c in sweep_eval_df.columns]

    meta_df = sweep_eval_df[["source_id"] + meta_cols].copy()
    meta_df["source_id"] = meta_df["source_id"].astype(str)

    if meta_df.columns.duplicated().any():
        dup_cols = meta_df.columns[meta_df.columns.duplicated()].tolist()
        print(f"Step {step}: dropping duplicate columns from meta_df:")
        print(dup_cols)
        meta_df = meta_df.loc[:, ~meta_df.columns.duplicated()].copy()

    # Remove stale metadata columns from out_df before merging clean metadata.
    for col in meta_cols:
        out_df = out_df.drop(columns=[col], errors="ignore")

    out_df = out_df.merge(meta_df, on="source_id", how="left")

    if out_df.columns.duplicated().any():
        dup_cols = out_df.columns[out_df.columns.duplicated()].tolist()
        raise RuntimeError(f"Step {step}: duplicate columns after metadata merge: {dup_cols}")

    if "reference_arabic" not in out_df.columns:
        if "target_arabic" in out_df.columns:
            out_df["reference_arabic"] = out_df["target_arabic"]
        else:
            raise RuntimeError("Missing both reference_arabic and target_arabic.")

    if "target_arabic" in out_df.columns:
        out_df["reference_arabic"] = (
            out_df["reference_arabic"]
            .fillna(out_df["target_arabic"])
            .astype(str)
        )
    else:
        out_df["reference_arabic"] = out_df["reference_arabic"].fillna("").astype(str)

    missing_count = out_df["prediction"].isna().sum()

    if missing_count > 0:
        out_df.to_csv(pred_path, index=False, encoding="utf-8-sig")
        raise RuntimeError(
            f"Step {step} is incomplete. Missing predictions: {missing_count}. "
            f"Rerun Cell 16 to continue from {pred_path}."
        )

    out_df["prediction"] = out_df["prediction"].fillna("").astype(str)
    out_df["reference_arabic"] = out_df["reference_arabic"].fillna("").astype(str)

    if "lora_draft" not in out_df.columns:
        raise RuntimeError("Missing lora_draft column in sweep output.")

    out_df["lora_draft"] = out_df["lora_draft"].fillna("").astype(str)

    empty_count = int((out_df["prediction"].str.strip().str.len() == 0).sum())

    if empty_count > 0:
        print(
            f"WARNING: step {step} has {empty_count} empty predictions. "
            "Metrics will count them as empty outputs."
        )

    keep_cols = [
        "source_id",
        "config",
        "domain",
        "dialect",
        "source_text",
        "lora_draft",
        "reference_arabic",
        "prediction",
        "draft_group",
        "garbage_draft",
        "empty_draft",
        "has_chinese",
        "mostly_non_arabic",
        "arabic_ratio",
        "han_ratio",
        "latin_ratio",
    ]

    keep_cols = [c for c in keep_cols if c in out_df.columns]
    out_df = out_df[keep_cols]
    out_df.to_csv(pred_path, index=False, encoding="utf-8-sig")

    # --------------------------------------------------------
    # Compute metrics
    # --------------------------------------------------------

    preds = out_df["prediction"].astype(str).tolist()
    refs = out_df["reference_arabic"].astype(str).tolist()
    drafts = out_df["lora_draft"].astype(str).tolist()

    metrics = compute_lexical_metrics(preds, refs)
    draft_metrics = compute_lexical_metrics(drafts, refs)

    changed_count = int(
        (
            out_df["lora_draft"].astype(str).str.strip()
            != out_df["prediction"].astype(str).str.strip()
        ).sum()
    )

    sweep_row = {
        "step": int(step),
        "checkpoint_dir": str(ckpt_dir),
        "corrector_adapter_dir": str(adapter_dir),
        "active_adapter_name": str(active_adapter_name),
        "prediction_path": str(pred_path),
        "n_examples": int(len(out_df)),
        "n_empty_predictions": int(empty_count),
        "changed_count": int(changed_count),
        "changed_rate": float(changed_count / max(1, len(out_df))),

        "draft_BLEU": float(draft_metrics["BLEU"]),
        "draft_spBLEU": float(draft_metrics["spBLEU"]),
        "draft_chrF": float(draft_metrics["chrF"]),
        "draft_chrF++": float(draft_metrics["chrF++"]),

        "BLEU": float(metrics["BLEU"]),
        "spBLEU": float(metrics["spBLEU"]),
        "chrF": float(metrics["chrF"]),
        "chrF++": float(metrics["chrF++"]),

        "delta_BLEU": float(metrics["BLEU"] - draft_metrics["BLEU"]),
        "delta_spBLEU": float(metrics["spBLEU"] - draft_metrics["spBLEU"]),
        "delta_chrF": float(metrics["chrF"] - draft_metrics["chrF"]),
        "delta_chrF++": float(metrics["chrF++"] - draft_metrics["chrF++"]),
    }

    # --------------------------------------------------------
    # Optional subset metrics: garbage vs non-garbage
    # --------------------------------------------------------

    if "garbage_draft" in out_df.columns:
        garbage_mask = out_df["garbage_draft"].astype(str).str.lower().isin(["true", "1"])

        for subset_name, mask in {
            "garbage": garbage_mask,
            "non_garbage": ~garbage_mask,
        }.items():
            subset_df = out_df[mask].copy()
            sweep_row[f"{subset_name}_n"] = int(len(subset_df))

            if len(subset_df) > 0:
                subset_pred_metrics = compute_lexical_metrics(
                    subset_df["prediction"].astype(str).tolist(),
                    subset_df["reference_arabic"].astype(str).tolist(),
                )

                subset_draft_metrics = compute_lexical_metrics(
                    subset_df["lora_draft"].astype(str).tolist(),
                    subset_df["reference_arabic"].astype(str).tolist(),
                )

                for k, v in subset_pred_metrics.items():
                    sweep_row[f"{subset_name}_{k}"] = float(v)
                    sweep_row[f"{subset_name}_draft_{k}"] = float(subset_draft_metrics[k])
                    sweep_row[f"{subset_name}_delta_{k}"] = float(v - subset_draft_metrics[k])

    sweep_rows.append(sweep_row)

    print("Metrics for step", step)
    print(json.dumps(sweep_row, ensure_ascii=False, indent=2))


# ------------------------------------------------------------
# Rank checkpoints and save sweep summary
# ------------------------------------------------------------

sweep_results_df = pd.DataFrame(sweep_rows)

if len(sweep_results_df) == 0:
    raise RuntimeError("No sweep results were produced.")

rank_by = []

for col in [PRIMARY_SELECTION_METRIC, SECONDARY_SELECTION_METRIC, "BLEU"]:
    if col in sweep_results_df.columns:
        rank_by.append(col)

if not rank_by:
    rank_by = ["chrF++", "spBLEU", "BLEU"]

sweep_results_ranked_df = sweep_results_df.sort_values(
    by=rank_by,
    ascending=[False] * len(rank_by),
).reset_index(drop=True)

sweep_results_path = sweep_dir / "lora_router_corrector_sweep_generation_metrics_steps_200_500_800_1000.csv"

sweep_results_ranked_df.to_csv(
    sweep_results_path,
    index=False,
    encoding="utf-8-sig",
)

display(sweep_results_ranked_df)

METRIC_BEST_STEP = int(sweep_results_ranked_df.iloc[0]["step"])
METRIC_BEST_CORRECTOR_ADAPTER_DIR = Path(sweep_results_ranked_df.iloc[0]["corrector_adapter_dir"])
METRIC_BEST_CHECKPOINT_DIR = Path(sweep_results_ranked_df.iloc[0]["checkpoint_dir"])

print("\nMetric-best step:", METRIC_BEST_STEP)
print("Metric-best checkpoint:", METRIC_BEST_CHECKPOINT_DIR)
print("Metric-best corrector adapter:", METRIC_BEST_CORRECTOR_ADAPTER_DIR)
print("Saved sweep results:", sweep_results_path)

Strict target sweep steps: [200, 500, 800, 1000]
Available corrector checkpoint steps: [100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650, 700, 750, 800, 850, 900, 950, 1000, 1050]
Final steps to sweep: [200, 500, 800, 1000]
Sweep eval examples: 1118
This is the same full eval set used by the LoRA draft baseline.

Sweeping LoRA router-corrector checkpoint step: 200
Checkpoint directory: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_lora_step700_alexandria_eg_only_context3_complete2shot_LORA_ROUTER_CORRECTOR_draft2gold_fulltrain_mix_badarabic_garbage_retain_worst0p3_chrfpp_fulleval_r8_alpha16_drop0p05_lr5e-05_3epochs/checkpoint-200
Adapter directory: /content/drive/MyDrive/alexandria_qwen35_sft/runs/qwen3_4b_lora_step700_alexandria_eg_only_context3_complete2shot_LORA_ROUTER_CORRECTOR_draft2gold_fulltrain_mix_badarabic_garbage_retain_worst0p3_chrfpp_fulleval_r8_alpha16_drop0p05_lr5e-05_3epochs/checkpoint-200
Prediction file: /content/drive/MyDrive/alexandria_qwen35_

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1596: UserWarning: Adapter router_corrector was active which is now deleted. Setting active adapter to default.
  warnings.warn(


Active adapter: router_corrector
Mode: cached-draft router-corrector, no adapter stacking
Trainable params: 0 / 2,271,870,464 (0.0000%)
Resuming step 200: 110 / 1118 already done.
Need to generate for step 200: 1008 remaining examples.


Generate step 200:   0%|          | 0/1008 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blo

In [ ]:
# ============================================================
# Cell 16B — Qualitative comparison: LoRA-700 draft vs LoRA router-correction
# ============================================================

import pandas as pd

if "METRIC_BEST_STEP" not in globals():
    raise RuntimeError("Run Cell 16 metric sweep first.")

BEST_SWEEP_PRED_PATH = (
    OUTPUT_DIR
    / "metric_sweep"
    / f"sweep_predictions_step{METRIC_BEST_STEP}.csv"
)

compare_df = pd.read_csv(BEST_SWEEP_PRED_PATH)
compare_df["lora_draft"] = compare_df["lora_draft"].fillna("").astype(str)
compare_df["prediction"] = compare_df["prediction"].fillna("").astype(str)
compare_df["reference_arabic"] = compare_df["reference_arabic"].fillna("").astype(str)

compare_df["changed"] = (
    compare_df["lora_draft"].str.strip()
    != compare_df["prediction"].str.strip()
)

print("Best step:", METRIC_BEST_STEP)
print("Rows:", len(compare_df))
print("Changed rows:", int(compare_df["changed"].sum()))

preview_cols = [
    "source_id",
    "draft_group",
    "garbage_draft",
    "source_text",
    "lora_draft",
    "prediction",
    "reference_arabic",
    "changed",
]
preview_cols = [c for c in preview_cols if c in compare_df.columns]

print("\nChanged examples preview:")
display(compare_df[compare_df["changed"]][preview_cols].head(20))

print("\nUnchanged examples preview:")
display(compare_df[~compare_df["changed"]][preview_cols].head(10))


In [ ]:
# ============================================================
# Cell 17 — Save/load metric-best LoRA router-corrector adapter
# ============================================================

import json
import shutil
from pathlib import Path

if "METRIC_BEST_CORRECTOR_ADAPTER_DIR" not in globals():
    raise RuntimeError("Run Cell 16 metric sweep first.")

CORRECTOR_DIR.mkdir(parents=True, exist_ok=True)

FINAL_CORRECTOR_PATH = CORRECTOR_DIR / f"lora_router_corrector_metric_best_step{METRIC_BEST_STEP}"
FINAL_CONFIG_PATH = CORRECTOR_DIR / f"lora_router_corrector_metric_best_step{METRIC_BEST_STEP}_config.json"

if FINAL_CORRECTOR_PATH.exists():
    print("Final metric-best adapter already exists; replacing the copy:", FINAL_CORRECTOR_PATH)
    shutil.rmtree(FINAL_CORRECTOR_PATH)

shutil.copytree(METRIC_BEST_CORRECTOR_ADAPTER_DIR, FINAL_CORRECTOR_PATH)

best_config = {
    "stage2_mode": STAGE2_MODE,
    "experiment_name": EXPERIMENT_NAME,
    "base_model": MODEL_NAME,
    "lora_experiment_name": LORA_EXPERIMENT_NAME,
    "lora_adapter_path": str(LORA_ADAPTER_PATH),
    "lora_best_step": int(LORA_BEST_STEP),
    "first_stage_adapter_name": FIRST_STAGE_ADAPTER_NAME,
    "corrector_adapter_name": CORRECTOR_ADAPTER_NAME,
    "metric_best_step": int(METRIC_BEST_STEP),
    "metric_best_checkpoint_dir": str(METRIC_BEST_CHECKPOINT_DIR),
    "metric_best_corrector_adapter_dir": str(METRIC_BEST_CORRECTOR_ADAPTER_DIR),
    "final_corrector_path": str(FINAL_CORRECTOR_PATH),
    "corrector_train_path": str(CORRECTOR_TRAIN_PATH),
    "diagnostic_counts_path": str(DIAGNOSTIC_COUNTS_PATH),
    "primary_selection_metric": PRIMARY_SELECTION_METRIC,
    "secondary_selection_metric": SECONDARY_SELECTION_METRIC,
    "corrector_lora_r": int(CORRECTOR_LORA_R),
    "corrector_lora_alpha": int(CORRECTOR_LORA_ALPHA),
    "corrector_lora_dropout": float(CORRECTOR_LORA_DROPOUT),
    "corrector_lr": float(CORRECTOR_LR),
    "corrector_num_epochs": int(CORRECTOR_NUM_EPOCHS),
}

FINAL_CONFIG_PATH.write_text(
    json.dumps(best_config, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

FINAL_CORRECTOR_ADAPTER_NAME = f"{CORRECTOR_ADAPTER_NAME}_metric_best"
load_corrector_adapter_for_inference(
    FINAL_CORRECTOR_PATH,
    adapter_name=FINAL_CORRECTOR_ADAPTER_NAME,
)

print("Saved metric-best LoRA router-corrector adapter:")
print("  from:", METRIC_BEST_CORRECTOR_ADAPTER_DIR)
print("  to:", FINAL_CORRECTOR_PATH)
print("Saved config:", FINAL_CONFIG_PATH)
print("Loaded final adapter for inference as:", FINAL_CORRECTOR_ADAPTER_NAME)


### **Full eval generation with LoRA router-corrector**


In [ ]:
# ============================================================
# Cell 18 — Full eval generation with metric-best LoRA router-corrector
# Resumable by source_id
# Second-stage corrector version: uses corrector_eval_df, not raw eval_df
# ============================================================

import pandas as pd
from tqdm.auto import tqdm

if "FINAL_CORRECTOR_PATH" not in globals():
    raise RuntimeError("Run Cell 17 first to save/load the metric-best LoRA router-corrector.")

if "METRIC_BEST_STEP" not in globals():
    raise RuntimeError("Run Cell 16 first to select METRIC_BEST_STEP.")

if "corrector_eval_df" not in globals():
    corrector_eval_df = pd.read_csv(EVAL_DRAFT_PATH)

corrector_eval_df["source_id"] = corrector_eval_df["source_id"].astype(str)

if len(corrector_eval_df) != len(eval_df):
    raise RuntimeError(
        "corrector_eval_df is not full eval. "
        f"corrector_eval_df={len(corrector_eval_df)}, eval_df={len(eval_df)}. "
        "Rerun Cell 9B."
    )

required_cols = {
    "source_id",
    "source_text",
    "target_arabic",
    "lora_draft",
    "few_shot_block",
    "metadata_block",
    "previous_context_block",
}

missing_cols = required_cols - set(corrector_eval_df.columns)

if missing_cols:
    raise ValueError(
        f"corrector_eval_df is missing required second-stage columns: {missing_cols}. "
        "Rerun Cell 9B."
    )

if "FINAL_CORRECTOR_ADAPTER_NAME" not in globals():
    FINAL_CORRECTOR_ADAPTER_NAME = f"{CORRECTOR_ADAPTER_NAME}_metric_best"

load_corrector_adapter_for_inference(
    FINAL_CORRECTOR_PATH,
    adapter_name=FINAL_CORRECTOR_ADAPTER_NAME,
)

EVAL_TAG = f"fulleval_lora_router_corrector_metric_best_step{METRIC_BEST_STEP}"

FULL_PRED_PATH = PRED_DIR / f"full_eval_predictions_{EXPERIMENT_NAME}_{EVAL_TAG}.csv"

print("Generating full eval predictions")
print("Experiment:", EXPERIMENT_NAME)
print("Corrector adapter:", FINAL_CORRECTOR_PATH)
print("Output:", FULL_PRED_PATH)

full_eval_df = corrector_eval_df.reset_index(drop=True).copy()
expected_ids = set(full_eval_df["source_id"].astype(str).tolist())

if FULL_PRED_PATH.exists():
    existing_df = pd.read_csv(FULL_PRED_PATH)
    existing_df["source_id"] = existing_df["source_id"].astype(str)

    existing_df = existing_df[
        existing_df["source_id"].isin(expected_ids)
    ].copy()

    existing_df = existing_df.drop_duplicates(
        subset=["source_id"],
        keep="first",
    )

    if "prediction" not in existing_df.columns:
        existing_df["prediction"] = ""

    existing_df["prediction"] = existing_df["prediction"].fillna("").astype(str)

    done_df = existing_df[
        existing_df["prediction"].str.strip().str.len() > 0
    ].copy()

    pred_rows = done_df.to_dict("records")
    done_ids = set(done_df["source_id"].astype(str).tolist())

    print(f"Resuming existing predictions: {len(done_ids)} / {len(full_eval_df)}")

else:
    pred_rows = []
    done_ids = set()
    print("No existing prediction file. Starting from scratch.")

for _, row in tqdm(
    full_eval_df.iterrows(),
    total=len(full_eval_df),
    desc="Full eval generation",
):
    row_dict = row.to_dict()
    source_id = str(row_dict["source_id"])

    if source_id in done_ids:
        continue

    try:
        pred = generate_with_lora_corrector_from_row(row_dict)
    except Exception as e:
        pred = ""
        print("Generation failed:", source_id, repr(e))

    pred_rows.append({
        "source_id": source_id,
        "config": row_dict.get("config", ""),
        "split": row_dict.get("split", ""),
        "conversation_id": row_dict.get("conversation_id", ""),
        "turn_id": row_dict.get("turn_id", ""),
        "country": row_dict.get("country", ""),
        "dialect": row_dict.get("dialect", ""),
        "domain": row_dict.get("domain", ""),
        "speaker": row_dict.get("speaker", ""),
        "gender_direction": row_dict.get("gender_direction", ""),
        "source_text": row_dict.get("source_text", ""),
        "lora_draft": row_dict.get("lora_draft", ""),
        "reference_arabic": row_dict.get("target_arabic", ""),
        "prediction": pred,
        "draft_group": row_dict.get("draft_group", ""),
        "garbage_draft": row_dict.get("garbage_draft", False),
        "empty_draft": row_dict.get("empty_draft", False),
        "has_chinese": row_dict.get("has_chinese", False),
        "mostly_non_arabic": row_dict.get("mostly_non_arabic", False),
        "arabic_ratio": row_dict.get("arabic_ratio", ""),
        "han_ratio": row_dict.get("han_ratio", ""),
        "latin_ratio": row_dict.get("latin_ratio", ""),
    })

    done_ids.add(source_id)

    if len(pred_rows) % PRED_SAVE_EVERY == 0:
        tmp_df = pd.DataFrame(pred_rows)
        tmp_df["source_id"] = tmp_df["source_id"].astype(str)
        tmp_df = tmp_df.drop_duplicates(subset=["source_id"], keep="first")
        tmp_df.to_csv(FULL_PRED_PATH, index=False, encoding="utf-8-sig")
        print(f"Saved progress: {len(done_ids)} / {len(full_eval_df)}")

final_pred_df = pd.DataFrame(pred_rows)

if len(final_pred_df) == 0:
    raise RuntimeError("No full-eval predictions generated or loaded.")

final_pred_df["source_id"] = final_pred_df["source_id"].astype(str)
final_pred_df = final_pred_df.drop_duplicates(subset=["source_id"], keep="first")

order_df = full_eval_df[["source_id"]].copy()
order_df["source_id"] = order_df["source_id"].astype(str)

final_pred_df = order_df.merge(final_pred_df, on="source_id", how="left")

meta_cols = [
    "config",
    "split",
    "conversation_id",
    "turn_id",
    "country",
    "dialect",
    "domain",
    "speaker",
    "gender_direction",
    "source_text",
    "lora_draft",
    "target_arabic",
    "draft_group",
    "garbage_draft",
    "empty_draft",
    "has_chinese",
    "mostly_non_arabic",
    "arabic_ratio",
    "han_ratio",
    "latin_ratio",
]

meta_cols = [c for c in meta_cols if c in full_eval_df.columns]
meta_df = full_eval_df[["source_id"] + meta_cols].copy()
meta_df["source_id"] = meta_df["source_id"].astype(str)

for col in meta_cols:
    final_pred_df = final_pred_df.drop(columns=[col], errors="ignore")

final_pred_df = final_pred_df.merge(meta_df, on="source_id", how="left")

if "reference_arabic" not in final_pred_df.columns:
    final_pred_df["reference_arabic"] = final_pred_df["target_arabic"]

final_pred_df["reference_arabic"] = (
    final_pred_df["reference_arabic"]
    .fillna(final_pred_df["target_arabic"])
    .astype(str)
)

final_pred_df["prediction"] = final_pred_df["prediction"].fillna("").astype(str)
final_pred_df["lora_draft"] = final_pred_df["lora_draft"].fillna("").astype(str)

keep_cols = [
    "source_id",
    "config",
    "split",
    "conversation_id",
    "turn_id",
    "country",
    "dialect",
    "domain",
    "speaker",
    "gender_direction",
    "source_text",
    "lora_draft",
    "reference_arabic",
    "prediction",
    "draft_group",
    "garbage_draft",
    "empty_draft",
    "has_chinese",
    "mostly_non_arabic",
    "arabic_ratio",
    "han_ratio",
    "latin_ratio",
]

keep_cols = [c for c in keep_cols if c in final_pred_df.columns]
final_pred_df = final_pred_df[keep_cols]
final_pred_df.to_csv(FULL_PRED_PATH, index=False, encoding="utf-8-sig")

missing_predictions = final_pred_df["prediction"].isna().sum()
empty_predictions = (
    final_pred_df["prediction"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.len()
    == 0
).sum()

print("\nSaved full predictions:")
print(FULL_PRED_PATH)
print("Rows:", len(final_pred_df))
print("Missing predictions:", int(missing_predictions))
print("Empty predictions:", int(empty_predictions))

if len(final_pred_df) == len(eval_df):
    print("Full prediction coverage confirmed.")
else:
    print("WARNING: row count does not match eval_df.")


### **Full lexical metrics**


In [ ]:
# ============================================================
# Cell 19 — Full lexical metrics: BLEU, spBLEU, chrF, chrF++
# Draft baseline vs second-stage LoRA router-corrector
# Includes garbage-draft and non-garbage subset metrics when available
# ============================================================

import json
import pandas as pd
from sacrebleu.metrics import BLEU, CHRF

if "FULL_PRED_PATH" not in globals():
    raise RuntimeError("Run Cell 18 first.")

FULL_METRICS_PATH = PRED_DIR / f"full_eval_metrics_{EXPERIMENT_NAME}_{EVAL_TAG}.json"

pred_df = pd.read_csv(FULL_PRED_PATH)

required_cols = {
    "source_id",
    "lora_draft",
    "prediction",
    "reference_arabic",
}

missing_cols = required_cols - set(pred_df.columns)

if missing_cols:
    raise ValueError(f"Missing required prediction columns: {missing_cols}")

pred_df["source_id"] = pred_df["source_id"].astype(str)
pred_df["lora_draft"] = pred_df["lora_draft"].fillna("").astype(str)
pred_df["prediction"] = pred_df["prediction"].fillna("").astype(str)
pred_df["reference_arabic"] = pred_df["reference_arabic"].fillna("").astype(str)

expected_eval_df = eval_df.reset_index(drop=True).copy()
expected_eval_df["source_id"] = expected_eval_df["source_id"].astype(str)

expected_ids = set(expected_eval_df["source_id"])
actual_ids = set(pred_df["source_id"])

missing_ids = expected_ids - actual_ids

print("Expected examples:", len(expected_eval_df))
print("Prediction rows:", len(pred_df))
print("Missing IDs:", len(missing_ids))

if missing_ids:
    raise RuntimeError("Prediction file is incomplete. Rerun Cell 18.")

order_df = expected_eval_df[["source_id"]].copy()

pred_df_ordered = order_df.merge(
    pred_df,
    on="source_id",
    how="left",
)

# Backfill subset flags from corrector_eval_df if the prediction file was resumed
# from an older version that lacked them.
if "garbage_draft" not in pred_df_ordered.columns and "corrector_eval_df" in globals():
    diag_cols = [
        "source_id",
        "garbage_draft",
        "empty_draft",
        "has_chinese",
        "mostly_non_arabic",
        "draft_group",
        "arabic_ratio",
        "han_ratio",
        "latin_ratio",
    ]
    diag_cols = [c for c in diag_cols if c in corrector_eval_df.columns]

    if "garbage_draft" in diag_cols:
        diag_df = corrector_eval_df[diag_cols].copy()
        diag_df["source_id"] = diag_df["source_id"].astype(str)
        pred_df_ordered = pred_df_ordered.merge(diag_df, on="source_id", how="left")


def compute_full_lexical_metrics(preds, refs):
    bleu_metric = BLEU(tokenize="13a")
    spbleu_metric = BLEU(tokenize="flores200")
    chrf_metric = CHRF(word_order=0)
    chrfpp_metric = CHRF(word_order=2)

    return {
        "BLEU": float(bleu_metric.corpus_score(preds, [refs]).score),
        "spBLEU": float(spbleu_metric.corpus_score(preds, [refs]).score),
        "chrF": float(chrf_metric.corpus_score(preds, [refs]).score),
        "chrF++": float(chrfpp_metric.corpus_score(preds, [refs]).score),
    }


def add_prefixed_metrics(out, prefix, preds, refs):
    metrics = compute_full_lexical_metrics(preds, refs)
    for k, v in metrics.items():
        out[f"{prefix}_{k}"] = v
    return metrics


def as_bool_series(series):
    return series.astype(str).str.lower().isin(["true", "1", "yes", "y"])


draft_preds = pred_df_ordered["lora_draft"].fillna("").astype(str).tolist()
corrector_preds = pred_df_ordered["prediction"].fillna("").astype(str).tolist()
refs = pred_df_ordered["reference_arabic"].fillna("").astype(str).tolist()

draft_metrics = compute_full_lexical_metrics(draft_preds, refs)
corrector_metrics = compute_full_lexical_metrics(corrector_preds, refs)

changed_count = int(
    (
        pred_df_ordered["lora_draft"].fillna("").astype(str).str.strip()
        != pred_df_ordered["prediction"].fillna("").astype(str).str.strip()
    ).sum()
)

full_metrics = {
    "system": f"Qwen3-4B-LoRA-step{LORA_BEST_STEP}+LoRA-router-corrector",
    "experiment_name": EXPERIMENT_NAME,
    "base_model": MODEL_NAME,
    "lora_experiment_name": LORA_EXPERIMENT_NAME,
    "lora_adapter_path": str(LORA_ADAPTER_PATH),
    "lora_best_step": int(LORA_BEST_STEP),
    "stage2_mode": STAGE2_MODE,
    "final_corrector_path": str(FINAL_CORRECTOR_PATH),
    "metric_best_step": int(METRIC_BEST_STEP),
    "corrector_selection_metric": CORRECTOR_SELECTION_METRIC,
    "corrector_bad_fraction": CORRECTOR_BAD_FRACTION,
    "corrector_mix_bad_arabic": CORRECTOR_MIX_BAD_ARABIC,
    "corrector_mix_garbage": CORRECTOR_MIX_GARBAGE,
    "corrector_mix_retain": CORRECTOR_MIX_RETAIN,
    "corrector_train_path": str(CORRECTOR_TRAIN_PATH),
    "corrector_train_examples": int(len(corrector_train_df)) if "corrector_train_df" in globals() else None,
    "n_examples": len(corrector_preds),
    "changed_count": changed_count,
    "changed_rate": changed_count / max(1, len(corrector_preds)),

    "draft_BLEU": draft_metrics["BLEU"],
    "draft_spBLEU": draft_metrics["spBLEU"],
    "draft_chrF": draft_metrics["chrF"],
    "draft_chrF++": draft_metrics["chrF++"],

    "corrector_BLEU": corrector_metrics["BLEU"],
    "corrector_spBLEU": corrector_metrics["spBLEU"],
    "corrector_chrF": corrector_metrics["chrF"],
    "corrector_chrF++": corrector_metrics["chrF++"],

    "delta_BLEU": corrector_metrics["BLEU"] - draft_metrics["BLEU"],
    "delta_spBLEU": corrector_metrics["spBLEU"] - draft_metrics["spBLEU"],
    "delta_chrF": corrector_metrics["chrF"] - draft_metrics["chrF"],
    "delta_chrF++": corrector_metrics["chrF++"] - draft_metrics["chrF++"],
}

# ------------------------------------------------------------
# Subset metrics: garbage drafts vs non-garbage drafts
# ------------------------------------------------------------

if "garbage_draft" in pred_df_ordered.columns:
    garbage_mask = as_bool_series(pred_df_ordered["garbage_draft"].fillna(False))

    for subset_name, mask in {
        "garbage": garbage_mask,
        "non_garbage": ~garbage_mask,
    }.items():
        subset_df = pred_df_ordered[mask].copy()
        full_metrics[f"{subset_name}_n"] = int(len(subset_df))

        if len(subset_df) == 0:
            continue

        subset_refs = subset_df["reference_arabic"].fillna("").astype(str).tolist()
        subset_draft_preds = subset_df["lora_draft"].fillna("").astype(str).tolist()
        subset_corrector_preds = subset_df["prediction"].fillna("").astype(str).tolist()

        subset_draft_metrics = compute_full_lexical_metrics(subset_draft_preds, subset_refs)
        subset_corrector_metrics = compute_full_lexical_metrics(subset_corrector_preds, subset_refs)

        for k, v in subset_draft_metrics.items():
            full_metrics[f"{subset_name}_draft_{k}"] = v

        for k, v in subset_corrector_metrics.items():
            full_metrics[f"{subset_name}_corrector_{k}"] = v
            full_metrics[f"{subset_name}_delta_{k}"] = v - subset_draft_metrics[k]

        subset_changed = int(
            (
                subset_df["lora_draft"].fillna("").astype(str).str.strip()
                != subset_df["prediction"].fillna("").astype(str).str.strip()
            ).sum()
        )
        full_metrics[f"{subset_name}_changed_count"] = subset_changed
        full_metrics[f"{subset_name}_changed_rate"] = subset_changed / max(1, len(subset_df))
else:
    full_metrics["subset_metrics_note"] = "garbage_draft column unavailable; subset metrics skipped."

FULL_METRICS_PATH.write_text(
    json.dumps(full_metrics, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(json.dumps(full_metrics, indent=2, ensure_ascii=False))
print("\nSaved full metrics:")
print(FULL_METRICS_PATH)
